# 📓 Notebook 1 : Ingestion, Cadrage et Couche Bronze

## 🛠️ Partie 0 : Cadrage, Périmètre et Configuration Globale

### 🎯 Objectifs du Carnet
- **Mission :** Mettre en place un pipeline d'ingestion reproductible pour la couche Bronze du Lakehouse `Lyon_lakehouse`.
- **Règles d'ingénierie appliquées :**
  1. **Traçabilité :** Chaque source est enregistrée dans un registre (`sources_registry`) avec son URL d'origine. Chaque ligne ingérée reçoit les colonnes techniques `_source_url`, `_source_last_updated`, `_source_ttl`, `_source_version` et `_ingestion_ts`.
  2. **Bronze fidèle :** Aucune donnée brute n'est détruite ni filtrée destructivement en amont. Les colonnes techniques sont ajoutées sans altérer les données sources.
  3. **Contrôle par analyse préalable du flux :** Avant persistance, le flux est analysé (statut HTTP, structure JSON, présence des champs attendus, coordonnées non nulles et dans la bbox). Les contrôles réellement bloquants sont explicités dans le code d'ingestion.


In [1]:
# Installation des dépendances géospatiales et utilitaires pour le pipeline
%pip install geopandas pyogrio shapely requests pyarrow folium

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 8, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.3/33.3 MB 164.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 150.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 153.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [folium]2m5/7 [geopandas]

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /nfs4/pyenv-84a54efc-6a56-439d-baf1-a3fee42b76c5/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



### 📝 : Importation des bibliothèques Python et rôle de chaque outil

- **Objectif :** Importer les packages fondamentaux nécessaires au traitement des données géographiques, des requêtes API et de l'ingestion Big Data.
- **Utilité des outils importés :**
  - `requests` : Permet d'effectuer les requêtes HTTP sécurisées vers les API ouvertes (GBFS, WFS) avec gestion des timeouts.
  - `pandas` : Manipule les DataFrames tabulaires légers pour le registre des sources et les petits tableaux de contrôle.
  - `geopandas` : Extension de Pandas indispensable pour manipuler les objets géospatiaux (points, polygones, systèmes de coordonnées).
  - `pyspark.sql` : Moteur de calcul distribué pour structurer les données et les persister dans le Lakehouse `Lyon_lakehouse` au format Delta.

In [2]:
# Importation des bibliothèques fondamentales du pipeline d'ingestion
import requests        # Pour les appels API sécurisés vers les sources ouvertes
import pandas as pd      # Pour la gestion du registre et des structures tabulaires
import geopandas as gpd  # Pour la manipulation et le filtrage des géométries spatiales
import folium            # Pour la cartographie interactive et la validation visuelle
from pyspark.sql import SparkSession # Pour piloter le moteur de calcul du Lakehouse
from pyspark.sql import functions as F # Pour les opérations Spark (filtres, transformations)
import json

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 10, Finished, Available, Finished, False)

### 📝 : Initialisation de la session Spark et des constantes globales du projet

- **Objectif :** connecter le carnet au Lakehouse `Lyon_lakehouse` et fixer une bonne fois pour toutes les constantes techniques pour éviter tout chiffre « magique » dans le code.
- **Paramètres fixés :**
  - `LAKEHOUSE_CIBLE` : le Lakehouse cible où seront écrites toutes nos tables Bronze.
  - `CRS_DEFAUT` (EPSG:4326) : système de coordonnées standard GPS (latitude/longitude) pour le stockage des objets géographiques.
  - `CRS_PROJET` (EPSG:2154) : Lambert 93, indispensable pour calculer des distances exactes en mètres (ex. accessibilité à 5 min à pied). Utilisé en Silver.
  - `MILLESIME_POP` (2021) : référence démographique unique (Filosofi 2021) pour éviter de mélanger des millésimes obsolètes.
  - `TIMEOUT_REQUETE` : (10, 60) secondes, appliqué à **tous** les `requests.get()` pour éviter les blocages réseau.
  - `BBOX_LYON` : boîte englobante **réservée au filtrage Silver**. Le Bronze conserve toutes les lignes, y compris celles hors Métropole, pour rester fidèle à la source.


In [3]:
# Initialisation de la session Spark pointant vers notre environnement
spark = SparkSession.builder.appName("Mobility_Lyon_Bronze").getOrCreate()

# -----------------------------------------------------------------------------
# Constantes et paramètres globaux du projet (source unique de vérité)
# -----------------------------------------------------------------------------
LAKEHOUSE_CIBLE = "Lyon_lakehouse"      # Lakehouse cible de toutes les tables Bronze
CRS_DEFAUT      = "EPSG:4326"            # WGS 84 — stockage géographique standard (lat/lon)
CRS_PROJET      = "EPSG:2154"            # Lambert 93 — calculs métriques (utilisé en Silver)
MILLESIME_POP   = 2021                   # Millésime de référence unique (Filosofi 2021)

# Timeout (connexion, lecture) en secondes — à passer à TOUS les requests.get()
# pour qu'une API qui ne répond pas ne bloque pas indéfiniment le notebook.
TIMEOUT_REQUETE = (10, 60)

# BBOX_LYON : réservé au filtrage de la couche Silver.
# Le Bronze conserve toutes les lignes (règle « Bronze fidèle »).
BBOX_LYON = {
    "lat_min": 45.65,
    "lat_max": 45.85,
    "lon_min": 4.70,
    "lon_max": 5.00
}

print(f"Session Spark active. Prêt à écrire dans le Lakehouse : {LAKEHOUSE_CIBLE}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 11, Finished, Available, Finished, False)

Session Spark active. Prêt à écrire dans le Lakehouse : Lyon_lakehouse


### 📝: Validation de la configuration et des constantes

- **Objectif :** Vérifier que l'environnement Spark est initialisé et que les paramètres globaux du projet sont prêts.
- **Note :** Les contrôles de qualité bloquants (timeouts, structure des données) seront appliqués directement lors de l'ingestion de chaque source dans les parties suivantes.

In [4]:
# Configuration de base validée. 
# Les contrôles stricts se feront au moment de l'ingestion des données.
print(f"✅ Partie 0 initialisée. Environnement prêt pour le Lakehouse '{LAKEHOUSE_CIBLE}'.")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 12, Finished, Available, Finished, False)

✅ Partie 0 initialisée. Environnement prêt pour le Lakehouse 'Lyon_lakehouse'.


# Partie 1 : Registre des Sources et Justification Métier

## 🎯 1. Objectif de la Démarche
Cette première partie pose les fondations fonctionnelles et architecturales de notre pipeline d'ingestion. À ce stade, l'objectif n'est pas de figer un dictionnaire technique rigide (schémas, formats de fichiers locaux ou noms de colonnes précis) que nous ne pourrions pas valider, mais d'établir un **cahier des charges intentionnel**. 

Nous définissons ici :
- Les **familles de données cibles** pour cartographier la mobilité à Lyon, en distinguant nettement l'offre de micromobilité, les infrastructures de transport lourd, les équipements de voirie, les points d'intérêt stratégiques et le cadre réglementaire.
- Les **types de données recherchés** au sein de chaque famille.
- La **justification métier** de chaque choix d'analyse.
- Le **mode d'ingestion privilégié** : aucune donnée ne doit transiter par un téléchargement manuel sur un poste local. Tout doit être récupéré dynamiquement et automatisé à la source (via des requêtes API et des flux en ligne) afin d'assurer la fraîcheur, l'évolutivité et la reproductibilité du pipeline.

---

## 📂 2. Taxonomie et Expression des Besoins par Famille de Données

### A. Famille : Offres de Mobilité Douce & Micromobilité
- **Types de données cibles :**
  * Les flottes de véhicules partagés en station fixe ou en libre-service (*free-floating* : vélos, trottinettes).
  * Les stations de vélos en libre-service et leurs disponibilités en temps réel.
- **Justification métier :** Capturer l'écosystème des mobilités douces de premier et dernier kilomètres pour analyser les usages de proximité, les zones de chalandise et la complémentarité avec le réseau lourd.
- **Stratégie d'ingestion :** Automatisation complète via des requêtes sur des **API normalisées (ex. GBFS)** pour capter la dynamique des flottes de micromobilité.

### B. Famille : Offres de Transport Lourd & Structurant (Réseau TCL)
- **Types de données cibles :**
  * Le réseau de transport en commun lourd et régulier : parcs relais, lignes de métro et funiculaire, tramways, lignes de bus et arrêts/stations du réseau.
- **Justification métier :** Centraliser l'ossature lourde et la maille fine de transport public pour mesurer la multimodalité, évaluer l'accessibilité globale et identifier les grands flux pendulaires de la métropole lyonnaise.
- **Stratégie d'ingestion :** Interrogation programmatique des services web ouverts et des flux WFS de la métropole/SYTRAL pour alimenter la couche Bronze du Lakehouse (à l'exclusion volontaire, pour cette première version, des modes secondaires ou à faible rendement pendulaire comme le réseau fluvial).

### C. Famille : Infrastructures et Aménagements de Voirie & Stationnement
- **Types de données cibles :**
  * Les parkings publics et parcs d'ouvrages institutionnels.
  * Les infrastructures de recharge pour véhicules électriques (IRVE).
  * Les aménagements cyclables, pistes et voiries dédiées aux mobilités douces.
- **Justification métier :** Évaluer le niveau d'équipement du territoire pour le stationnement automobile, la transition énergétique (bornes électriques) et la sécurité des déplacements doux.
- **Stratégie d'ingestion :** Connexion automatisée aux services cartographiques et portails OpenData (via requêtes programmatiques) pour récupérer des flux géospatiaux à jour.

### D. Famille : Socio-démographie et Population
- **Types de données cibles :**
  * Le maillage spatial carroyé (données de population, de revenus et de motorisation des ménages).
  * Les données démographiques globales et prospectives par secteur.
- **Justification métier :** Croiser l'offre de transport (lourd et micromobilité) avec la réalité humaine. Cela permet d'identifier les zones de forte tension, de mesurer l'adéquation entre l'offre et les besoins réels des habitants, et de repérer les zones blanches.
- **Stratégie d'ingestion :** Récupération programmatique de fichiers statistiques et géographiques de référence (ex. données INSEE Filosofi) directement depuis les plateformes officielles.

### E. Famille : Points d'Intérêt (POI) Stratégiques & Économiques
- **Types de données cibles :**
  * Les structures de santé et établissements hospitaliers.
  * Le tissu économique et les établissements actifs (base SIRENE / entreprises, zones d'activités).
  * Les pôles commerciaux et zones de chalandise de proximité.
- **Justification métier :** Identifier les grands bassins d'attractivité quotidienne (foyers d'emploi, flux de patients/visiteurs, commerces) qui génèrent des besoins de mobilité récurrents et justifient l'implantation stratégique des stations partagées.
- **Stratégie d'ingestion :** Interrogation programmatique des API et bases de données publiques géolocalisées (ex. données Sirene de l'INSEE, bases hospitalières ou OpenStreetMap géofensées sur la Métropole de Lyon) pour intégrer ces gisements de flux directement dans la couche Bronze.

### F. Famille : Cadre Réglementaire et Environnemental
- **Types de données cibles :**
  * Le périmètre des Zones à Faibles Émissions (ZFE).
  * Les limites administratives officielles de la Métropole de Lyon.
- **Justification métier :** Intégrer les contraintes réglementaires qui influencent directement les choix de déplacement des citoyens et justifient le développement des alternatives partagées et propres.
- **Stratégie d'ingestion :** Requêtage de couches géographiques officielles pour intégrer les polygones de délimitation réglementaire.

---

## 🔌 3. Principes Directeurs pour l'Ingestion (Couche Bronze)
- **Zéro manipulation locale (avec exception documentée) :** Les scripts Python s'appuient sur la bibliothèque `requests` pour interroger directement les sources distantes. Aucun fichier n'est téléchargé manuellement sur une machine hôte puis versé dans le Lakehouse.
  - **Exception :** Le fichier CSV relatif à la motorisation des ménages (Famille D) est déposé manuellement, faute d'endpoint API direct. Il sera tracé dans le registre avec sa date de dépôt et son URL d'origine.
- **Fraîcheur et Résilience :** En privilégiant les points d'accès dynamiques (API/flux), le pipeline garantit que toute exécution reflète l'état le plus récent de la donnée, s'affranchissant ainsi de l'obsolescence rapide des fichiers statiques.

---


## Partie 2 : Ingestion et Visualisation de la Couche Bronze (Données Brutes)

Dans cette partie, nous procédons à l'ingestion automatisée de l'ensemble de nos sources de données directement depuis leurs points d'accès officiels (API, flux ou services web), sans aucune modification ni nettoyage. 

**Objectifs de cette étape :**
1. **Extraction et Persistance :** Récupérer la donnée brute et l'enregistrer en l'état dans le Lakehouse (couche Bronze) pour garantir la traçabilité.
2. **Inspection visuelle :** Afficher un aperçu tabulaire (premières lignes) et géospatial (cartes Folium) pour s'assurer de la bonne récupération et de la cohérence spatiale des données lyonnaises.

### 2.1 Ingestion et Visualisation des flux de micromobilité (Couche Bronze)
Dans cette partie, nous procédons à l'ingestion des données relatives à la **micromobilité à Lyon** (vélos en libre-service, trottinettes, etc.). 

### Pourquoi isoler la micromobilité ?
Contrairement aux transports en commun lourds ou cadencés, la micromobilité répond à des logiques de déplacement plus spontanées, souvent caractérisées par des flux de type "dernier kilomètre" et une forte variabilité spatio-temporelle. Pour bien appréhender ces usages, nous nous appuyons ici sur des flux spécifiques et dédiés à ces modes doux.

**Objectifs de cette étape :**
1. **Chargement des flux bruts :** Ingestion de l'ensemble des flux de données de micromobilité dans la couche Bronze, **sans aucun croisement de données ni analyse** à ce stade.
2. **Inspection visuelle et structurelle :** Affichage des en-têtes, des premières lignes et visualisation spatiale (via Folium) pour analyser en détail la structure brute et la cohérence géographique de ces données.
3. **Recommandations pour la couche Silver :** Établir des pistes et des recommandations concrètes basées sur cette inspection structurelle en vue du prochain nettoyage.


### 🔍 Analyse et Spécifications de la Source : Citiz (GBFS v3.0 - Station Information)

* **Fournisseur / Opérateur :** Citiz / Métropole de Lyon
* **URL de l'API :** `https://backend.citiz.fr/public/provider/3/gbfs/v3.0/station_information.json`
* **Format d'entrée :** JSON (Norme ouverte GBFS v3.0)
* **Indicateurs de fraîcheur (Métadonnées du flux) :** 
  * `version` : 3.0
  * `ttl` : 300 secondes (durée de validité recommandée avant rafraîchissement)
  * `last_updated` : Horodatage dynamique de la dernière mise à jour par l'opérateur

#### 🌳 Topographie et Structure du Payload JSON
* **Chemin racine des données :** `data -> stations` (liste d'objets JSON représentant chaque station physique)
* **Exhaustivité des attributs bruts (Colonnes natives) :**
  * `station_id` (Identifiant unique de la station)
  * `name` (Nom commercial et emplacement)
  * `lat` / `lon` (Coordonnées géographiques WGS84)
  * `address` (Adresse postale descriptive)
  * `parking_type` (Type de stationnement : ex. `underground_parking`, `street_parking`, `parking_lot`)
  * `capacity` (Capacité maximale en véhicules)
  * `is_virtual_station` (Booléen indiquant s'il s'agit d'une station virtuelle)

#### ⚠️ Points de Vigilance Techniques (Pour la future Couche Silver) :
* **Particularité du champ `name` :** Le champ n'est pas une simple chaîne de caractères mais une liste imbriquée de dictionnaires multilingues (ex: `[{"text": "Lyon 1er - Parking LPA Terreaux", "language": "fr"}]`). Une extraction ciblée sera nécessaire lors du nettoyage ultérieur en couche Silver.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
* **Mode de collecte :** Requête HTTP GET automatisée via la bibliothèque Python `requests` avec gestion d'un **timeout** strict. 
* **Principe :** Les données brutes sont persistées telles quelles. Les métadonnées du flux (`last_updated`, `ttl`, `version`) et l'URL source sont ajoutées comme colonnes techniques pour garantir une traçabilité totale.


In [5]:
# 1. Requête HTTP GET vers l'API Citiz avec timeout
url_cityz_lyon = "https://backend.citiz.fr/public/provider/3/gbfs/v3.0/station_information.json"

# Application du timeout défini dans la Partie 0
response = requests.get(url_cityz_lyon, timeout=TIMEOUT_REQUETE)

# Contrôle qualité bloquant : lève une exception si le statut HTTP n'est pas 200
response.raise_for_status()

# 2. Conversion du texte brut de la réponse en dictionnaire Python
donnees_json = response.json()

# Extraction ciblée des stations
cityz_stations_list = donnees_json.get("data", {}).get("stations", [])
df_stations_cityz = pd.DataFrame(cityz_stations_list)

# 3. Ajout des colonnes de traçabilité (Règle d'or de la couche Bronze)
df_stations_cityz["_source_url"] = url_cityz_lyon
df_stations_cityz["_source_last_updated"] = donnees_json.get("last_updated")
df_stations_cityz["_source_ttl"] = donnees_json.get("ttl")
df_stations_cityz["_source_version"] = donnees_json.get("version")
df_stations_cityz["_ingestion_ts"] = pd.Timestamp.utcnow()

print(f"🎉 Succès ! {len(df_stations_cityz)} stations Citiz chargées à l'état brut (Couche Bronze).")

# Affichage d'un aperçu des colonnes clés
display(df_stations_cityz[["station_id", "name", "capacity", "lon", "lat", "address"]].head())

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 13, Finished, Available, Finished, False)

🎉 Succès ! 200 stations Citiz chargées à l'état brut (Couche Bronze).


SynapseWidget(Synapse.DataFrame, 641950bb-9037-400c-a59f-d1ff8bce1305)

### 🎉 Résultat de l'ingestion brute (Couche Bronze) - Citiz

* **Succès du chargement :** Les stations Citiz ont été extraites avec succès et converties en DataFrame Pandas brut, prêtes pour l'analyse spatiale et la persistance.
* **Point d'attention (Couche Silver) :** Le champ `name` présente une structure imbriquée de dictionnaires multilingues (ex. `[{"text": "...", "language": "fr"}]`). Un traitement spécifique d'extraction ciblée sera obligatoire lors du passage à la couche Silver pour le rendre exploitable.

### 🗺️ Validation Spatiale : Cartographie interactive des stations Citiz

Avant de procéder à la persistance définitive des données dans la couche Bronze, nous effectuons une validation visuelle et spatiale via une carte interactive.

* **Fond de carte :** Utilisation d'un fond Esri neutre (`World Light Gray Base`) pour mettre en valeur les données sans interférence visuelle excessive.
* **Représentation graphique :** Chaque station géolocalisée est matérialisée par un marqueur circulaire (`CircleMarker`).
* **Interactivité :** Une infobulle (popup HTML) au clic permet de consulter instantanément les attributs clés de la station (`nom`, `ID`, `capacité`).

In [6]:
# 1. Initialisation de la carte interactive centrée sur le cœur de Lyon avec un fond Esri neutre
carte_lyon_citiz = folium.Map(
    location=[45.7640, 4.8357],  # Point central de Lyon : [Latitude, Longitude]
    zoom_start=13,               # Niveau de zoom initial pour englober la zone urbaine
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", # URL du serveur de tuiles géographiques neutres
    attr="Esri"                  # Mention légale obligatoire (attribution du fournisseur de la carte)
)

# 2. Parcours itératif ligne par ligne du DataFrame Pandas (utilisation de 'station' pour plus de lisibilité)
for index, station in df_stations_cityz.iterrows():
    
    # 3. Contrôle d'intégrité : on vérifie impérativement que les coordonnées GPS existent et ne sont pas vides (NaN)
    if pd.notna(station['lat']) and pd.notna(station['lon']):
        
        # 4. Structuration du contenu HTML de l'infobulle (popup) sur plusieurs lignes grâce aux triple guillemets
        info_bulle = f"""
        <b>Station :</b> {station.get('name', 'Inconnue')}<br>
        <b>ID :</b> {station.get('station_id', 'N/A')}<br>
        <b>Capacité :</b> {station.get('capacity', 'N/A')} véhicules
        """
        
        # 5. Création et ajout d'un marqueur circulaire stylisé pour chaque station valide
        folium.CircleMarker(
            location=[station['lat'], station['lon']], # Positionnement précis de la station
            radius=6,                                  # Taille du cercle en pixels
            color='#2b5c8f',                           # Couleur de la bordure du cercle
            weight=1,                                  # Épaisseur de la bordure
            fill=True,                                 # Active le remplissage de la forme
            fill_color='#3182ce',                      # Couleur de remplissage intérieure
            fill_opacity=0.8,                          # Transparence du remplissage
            popup=folium.Popup(info_bulle, max_width=300) # Intégration du bloc HTML interactif au clic
        ).add_to(carte_lyon_citiz)                     # Ajout de ce marqueur sur notre objet carte global

# 6. Affichage final de la carte interactive dans l'environnement Microsoft Fabric / Jupyter
display(carte_lyon_citiz)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 14, Finished, Available, Finished, False)

### 💾 Persistance des données brutes dans le Lakehouse (Couche Bronze)

Après validation spatiale sur la carte interactive, nous procédons à la sauvegarde des données brutes dans la table Delta de la couche Bronze.

* **Format de stockage :** Delta Lake (persistance orientée colonnes, ACID et versionnée).
* **Mode d'écriture :** `overwrite` (remplacement complet à chaque cycle d'ingestion brute).
* **Règle métier :** Les données enregistrées sont strictement identiques aux données brutes renvoyées par l'API Citiz, sans aucune transformation.

In [7]:
# 3. Ingestion dans le Lakehouse Spark/Delta (Couche Bronze)

# Nom de la table cible
table_name = "bronze_citiz_stations"

# Conversion du DataFrame Pandas en DataFrame Spark
spark_df_cityz = spark.createDataFrame(df_stations_cityz)

# Écriture de la table persistante dans le Lakehouse
(
    spark_df_cityz.write
    .format("delta")                
    .mode("overwrite")              
    .option("overwriteSchema", "true") # Ajouté pour autoriser l'évolution du schéma
    .saveAsTable(table_name)        
)

print(f"✅ Table Delta '{table_name}' enregistrée avec succès dans la couche Bronze du Lakehouse !")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 15, Finished, Available, Finished, False)

✅ Table Delta 'bronze_citiz_stations' enregistrée avec succès dans la couche Bronze du Lakehouse !


### 🔍 Analyse et Spécifications de la Source : Dott (GBFS v2.3 - Station Information)

- **Fournisseur / Opérateur :** Tier-Dott / Métropole de Lyon
- **URL de l'API :** `https://gbfs.api.ridedott.com/public/v2/lyon/station_information.json`
- **Format d'entrée :** JSON (Norme ouverte GBFS v2.3)
- **Indicateurs de fraîcheur et Gouvernance (Métadonnées de flux & data.gouv.fr) :** 
  * `version` : 2.3
  * `ttl` : 300 secondes (fréquence de rafraîchissement temps réel)
  * **Validation officielle :** Contrôlé via le validateur *MobilityData* sur data.gouv.fr (taux de disponibilité de 100 %, aucune erreur détectée).

#### 🌳 Topographie et Structure du Payload JSON
- **Chemin racine des données :** `data -> stations` (liste d'objets JSON représentant les emplacements de stationnement).
- **Spécificité topographique :** Les points répertoriés sont majoritairement des stations virtuelles (`is_virtual_station: true`).
- **Exhaustivité des attributs bruts (Colonnes natives) :**
  * `station_id` (Identifiant unique de la station)
  * `name` (Adresse ou description textuelle de l'emplacement)
  * `lat` / `lon` (Coordonnées géographiques WGS84)
  * `region_id` (Identifiant de la zone géographique / Métropole de Lyon)
  * `is_virtual_station` (Indicateur booléen de station virtuelle)
  * `vehicle_capacity` (Capacité et type de véhicules associés)

#### ⚠️ Points de Vigilance Techniques (Pour la future Couche Silver) :
- **Particularité du champ `vehicle_capacity` :** Contrairement à un entier brut, cet attribut est encapsulé dans un dictionnaire imbriqué (ex: `{"dott_scooter": 10}`). Un traitement d'aplatissement spécifique sera programmé lors de la transition vers la couche Silver.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** Requête HTTP GET automatisée via la bibliothèque Python `requests` directement depuis l'endpoint distant.
- **Principe :** Intégrité stricte. Les données brutes sont persistées telles quelles dans le Lakehouse (format Delta, mode `overwrite`) pour garantir l'immutabilité et l'auditabilité de la source.


In [8]:
# 1. Requête HTTP GET vers l'API Dott avec timeout
url_dott_lyon = "https://gbfs.api.ridedott.com/public/v2/lyon/station_information.json"

# Application du timeout défini dans la Partie 0
response = requests.get(url_dott_lyon, timeout=TIMEOUT_REQUETE)

# Contrôle qualité bloquant
response.raise_for_status()

# 2. Conversion en dictionnaire Python
donnees_json = response.json()

# Extraction ciblée des stations
dott_stations_list = donnees_json.get("data", {}).get("stations", [])
df_stations_dott = pd.DataFrame(dott_stations_list)

# 3. Ajout des colonnes de traçabilité (Couche Bronze)
df_stations_dott["_source_url"] = url_dott_lyon
df_stations_dott["_source_last_updated"] = donnees_json.get("last_updated")
df_stations_dott["_source_ttl"] = donnees_json.get("ttl")
df_stations_dott["_source_version"] = donnees_json.get("version")
df_stations_dott["_ingestion_ts"] = pd.Timestamp.utcnow()

print(f"🎉 Succès ! {len(df_stations_dott)} stations Dott chargées à l'état brut (Couche Bronze).")

# Affichage d'un aperçu
display(df_stations_dott[["is_virtual_station", "station_id", "name", "vehicle_capacity", "lon", "lat"]].head())

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 16, Finished, Available, Finished, False)

🎉 Succès ! 1379 stations Dott chargées à l'état brut (Couche Bronze).


SynapseWidget(Synapse.DataFrame, 986ab360-2370-421a-995d-d66f566076f8)

### 🎉 Résultat de l'ingestion brute (Couche Bronze) - Dott

* **Succès du chargement :** Les stations Dott ont été extraites avec succès et converties en DataFrame Pandas brut, prêtes pour l'analyse spatiale.
* **Point d'attention (Couche Silver) :** La colonne `vehicle_capacity` présente un dictionnaire imbriqué (ex. `{"dott_scooter": 10}`). Un traitement spécifique d'aplatissement sera obligatoire lors du passage à la couche Silver pour rendre cette capacité exploitable.

### 🗺️ Validation Spatiale : Cartographie interactive des stations Dott

Avant de procéder à la persistance définitive des données dans la couche Bronze, nous effectuons une validation visuelle et spatiale via une carte interactive.

* **Fond de carte :** Utilisation d'un fond Esri neutre (`World Light Gray Base`) pour mettre en valeur les données sans interférence visuelle excessive.
* **Représentation graphique :** Chaque station géolocalisée est matérialisée par un marqueur circulaire (`CircleMarker`).
* **Interactivité :** Une infobulle (popup HTML) au clic permet de consulter instantanément les attributs clés de la station (`nom`, `ID`, `capacité`).

In [9]:
# 1. Initialisation de la carte interactive centrée sur le cœur de Lyon avec un fond Esri neutre
carte_lyon_dott = folium.Map(
    location=[45.7640, 4.8357],
    zoom_start=13,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Parcours itératif ligne par ligne du DataFrame Pandas
for index, station in df_stations_dott.iterrows():
    
    # 3. Contrôle d'intégrité
    if pd.notna(station['lat']) and pd.notna(station['lon']):
        
        # 4. Structuration du contenu HTML de l'infobulle (Correction : vehicle_capacity)
        info_bulle = f"""
        <b>Station :</b> {station.get('name', 'Inconnue')}<br>
        <b>ID :</b> {station.get('station_id', 'N/A')}<br>
        <b>Capacité :</b> {station.get('vehicle_capacity', 'N/A')} véhicules
        """
        
        # 5. Création et ajout d'un marqueur circulaire
        folium.CircleMarker(
            location=[station['lat'], station['lon']],
            radius=6,
            color='#2b5c8f',
            weight=1,
            fill=True,
            fill_color='#3182ce',
            fill_opacity=0.8,
            popup=folium.Popup(info_bulle, max_width=300)
        ).add_to(carte_lyon_dott)  # <-- CORRECTION : carte_lyon_dott au lieu de carte_lyon_citiz

# 6. Affichage final de la carte Dott
display(carte_lyon_dott)  # <-- CORRECTION : afficher la bonne carte

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 17, Finished, Available, Finished, False)

### 💾 Persistance des données brutes dans le Lakehouse (Couche Bronze)

Après validation spatiale sur la carte interactive, nous procédons à la sauvegarde des données brutes dans la table Delta de la couche Bronze.

* **Format de stockage :** Delta Lake (persistance orientée colonnes, ACID et versionnée).
* **Mode d'écriture :** `overwrite` (remplacement complet à chaque cycle d'ingestion brute).
* **Règle métier :** Les données enregistrées sont strictement identiques aux données brutes renvoyées par l'API Dott, sans aucune transformation.

In [10]:
# Persistance des données brutes Dott dans le Lakehouse (Couche Bronze)

# On définit le nom de la table cible
table_name_dott = "bronze_dott_stations"

# Conversion du DataFrame Pandas en DataFrame Spark
spark_df_dott = spark.createDataFrame(df_stations_dott)

# Écriture de la table persistante dans le Lakehouse
(
    spark_df_dott.write
    .format("delta")                
    .mode("overwrite")              
    .option("overwriteSchema", "true") # Ajouté pour autoriser l'évolution du schéma
    .saveAsTable(table_name_dott)        
)

print(f"✅ Table Delta '{table_name_dott}' enregistrée avec succès dans la couche Bronze du Lakehouse !")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 18, Finished, Available, Finished, False)

✅ Table Delta 'bronze_dott_stations' enregistrée avec succès dans la couche Bronze du Lakehouse !


### 🔍 Analyse et Spécifications de la Source : Vélo'v (JCDecaux)

* **Fournisseur / Opérateur :** JCDecaux / Métropole de Lyon
* **URL de l'API :** `https://api.jcdecaux.com/vls/v3/stations?contract=lyon&apiKey=VOTRE_CLE_API` (ou le flux `jcd_jcdecaux.jcdvelov` correspondant)
* **Format d'entrée :** JSON
* **Indicateurs de fraîcheur :** Les données JCDecaux sont mises à jour en temps réel.

#### 🌳 Topographie et Structure du Payload JSON
* **Chemin racine des données :** Liste d'objets JSON (ou `data -> stations` selon la version du flux).
* **Exhaustivité des attributs bruts :**
  * `number` (Identifiant unique de la station)
  * `name` (Nom de la station)
  * `position` (Latitude et Longitude WGS84)
  * `available_bikes` (Vélos disponibles)
  * `available_bike_stands` (Places libres)
  * `status` (Statut de la station : OPEN/CLOSED)

#### ⚠️ Points de Vigilance Techniques (Couche Silver) :
* **Coordonnées aberrantes :** Risque de stations avec des coordonnées à `0.0` ou hors périmètre (ex: Paris).

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
* Requête HTTP GET avec timeout et ajout des colonnes de traçabilité (`_source_url`, `_ingestion_ts`).


In [11]:
# 1. Requête HTTP GET vers l'API Vélo'v / JCDecaux
url_velo_lyon = "https://download.data.grandlyon.com/files/rdata/jcd_jcdecaux.jcdvelov/station_information.json"
response = requests.get(url_velo_lyon, timeout=TIMEOUT_REQUETE)  # Timeout ajouté
response.raise_for_status()  # Lève une erreur si le statut HTTP est >= 400

# 2. Vérification du succès de la connexion (Code 200)
if response.status_code == 200:
    # 2.1 Conversion de la réponse brute en dictionnaire JSON
    donnees_json = response.json()
    
    # 2.2 Extraction ciblée de la liste des stations
    velo_stations_list = donnees_json.get("data", {}).get("stations", [])
    
    # 2.3 Création du DataFrame Pandas brut (Couche Bronze)
    df_stations_velo = pd.DataFrame(velo_stations_list)
    
    # 2.4 Ajout des colonnes de traçabilité (Règle d'or de la couche Bronze)
    df_stations_velo["_source_url"] = url_velo_lyon
    df_stations_velo["_source_last_updated"] = donnees_json.get("last_updated")
    df_stations_velo["_source_ttl"] = donnees_json.get("ttl")
    df_stations_velo["_source_version"] = donnees_json.get("version")
    df_stations_velo["_ingestion_ts"] = pd.Timestamp.utcnow()
    
    # 2.5 Affichage d'un message de succès
    print(f"🎉 Succès ! {len(df_stations_velo)} stations Vélo'v importées avec succès (Couche Bronze).")
    
    # 2.6 Aperçu visuel des colonnes clés
    display(df_stations_velo[["station_id", "name", "lat", "lon", "capacity"]].head())

else:
    print(f"❌ Erreur lors du chargement ! Code retour : {response.status_code}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 19, Finished, Available, Finished, False)

🎉 Succès ! 467 stations Vélo'v importées avec succès (Couche Bronze).


SynapseWidget(Synapse.DataFrame, 8ee000f4-96d4-4ab4-9ffc-a683d7a46e17)

### 📊 Résultat de l'analyse brute (Vélo'v)

L'inspection des données brutes met en évidence deux anomalies majeures à traiter impérativement lors du passage en couche Silver :
1. **Coordonnées à zéro :** Présence de stations avec des latitudes et longitudes à `0.0`. Ces points aberrants devront être écartés.
2. **Coordonnées hors périmètre :** Présence de stations avec des coordonnées spatialement aberrantes (ex: latitude `48.80`, longitude `1.95`), totalement hors de la Métropole de Lyon. Un filtrage géographique strict sera nécessaire.

### 🗺️ Validation Spatiale : Cartographie interactive des stations Vélo'v

Avant de procéder à la persistance définitive des données dans la couche Bronze, nous effectuons une validation visuelle et spatiale via une carte interactive.

* **Fond de carte :** Utilisation d'un fond Esri neutre (`World Light Gray Base`).
* **Représentation graphique :** Chaque station géolocalisée est matérialisée par un marqueur circulaire (`CircleMarker`).
* **Interactivité :** Une infobulle (popup HTML) au clic permet de consulter les attributs clés.

In [12]:
# 1. Initialisation de la carte interactive centrée sur le cœur de Lyon avec un fond Esri neutre
carte_lyon_velo = folium.Map(
    location=[45.7640, 4.8357],  # Point central de Lyon : [Latitude, Longitude]
    zoom_start=13,               # Niveau de zoom initial pour englober la zone urbaine
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", # URL du serveur de tuiles géographiques neutres
    attr="Esri"                  # Mention légale obligatoire (attribution du fournisseur de la carte)
)

# 2. Parcours itératif ligne par ligne du DataFrame Pandas (utilisation de 'station' pour plus de lisibilité)
for index, station in df_stations_velo.iterrows():
    
    # 3. Contrôle d'intégrité : on vérifie impérativement que les coordonnées GPS existent et ne sont pas vides (NaN)
    if pd.notna(station['lat']) and pd.notna(station['lon']):
        
        # 4. Structuration du contenu HTML de l'infobulle (popup) sur plusieurs lignes grâce aux triple guillemets
        info_bulle = f"""
        <b>Station :</b> {station.get('name', 'Inconnue')}<br>
        <b>ID :</b> {station.get('station_id', 'N/A')}<br>
        <b>Capacité :</b> {station.get('capacity', 'N/A')} véhicules
        """
        
        # 5. Création et ajout d'un marqueur circulaire stylisé pour chaque station valide
        folium.CircleMarker(
            location=[station['lat'], station['lon']], # Positionnement précis de la station
            radius=6,                                  # Taille du cercle en pixels
            color='#2b5c8f',                           # Couleur de la bordure du cercle
            weight=1,                                  # Épaisseur de la bordure
            fill=True,                                 # Active le remplissage de la forme
            fill_color='#3182ce',                      # Couleur de remplissage intérieure
            fill_opacity=0.8,                          # Transparence du remplissage
            popup=folium.Popup(info_bulle, max_width=300) # Intégration du bloc HTML interactif au clic
        ).add_to(carte_lyon_velo)                     # Ajout de ce marqueur sur notre objet carte global

# 6. Affichage final de la carte interactive dans l'environnement Microsoft Fabric / Jupyter
display(carte_lyon_velo)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 20, Finished, Available, Finished, False)

### 💾 Persistance des données brutes dans le Lakehouse (Couche Bronze)

Après validation spatiale sur la carte interactive, nous procédons à la sauvegarde des données brutes dans la table Delta de la couche Bronze.

* **Format de stockage :** Delta Lake (persistance orientée colonnes, ACID et versionnée).
* **Mode d'écriture :** `overwrite` (remplacement complet à chaque cycle d'ingestion brute).
* **Règle métier :** Les données enregistrées sont strictement identiques aux données brutes renvoyées par l'API Dott, sans aucune transformation.

In [13]:
# 3. Ingestion dans le Lakehouse Spark/Delta (Couche Bronze)

table_name = "bronze_velo_stations"

# Astuce Bronze : Convertir la colonne liste (rental_methods) en string pour éviter le conflit de type Spark
if "rental_methods" in df_stations_velo.columns:
    df_stations_velo["rental_methods"] = df_stations_velo["rental_methods"].astype(str)

# Conversion du DataFrame Pandas en DataFrame Spark
spark_df_velo = spark.createDataFrame(df_stations_velo)

# Écriture de la table persistante dans le Lakehouse avec fusion de schéma autorisée (.option("mergeSchema", "true"))
(
    spark_df_velo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .saveAsTable(table_name)
)

print(f"✅ Table Delta '{table_name}' enregistrée avec succès dans la couche Bronze du Lakehouse.")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 21, Finished, Available, Finished, False)

✅ Table Delta 'bronze_velo_stations' enregistrée avec succès dans la couche Bronze du Lakehouse.


### 🔍 Analyse et Spécifications de la Source : Leo&Go (GBFS v2.3 - Autopartage Free-Floating)

- **Fournisseur / Opérateur :** Leo&Go / Grand Lyon
- **URL de l'API / Fichier :** `https://download.data.grandlyon.com/files/rdata/lag_leoandgo.disponibilite/free_bike_status.json`
- **Format d'entrée :** JSON (Standard GBFS v2.3 - Flux Temps Réel)
- **Indicateurs de fraîcheur et Gouvernance :**
  * `ttl` : 30 secondes (haute fréquence d'actualisation)
  * `last_updated` : Horodatage UNIX dynamique

#### 🌳 Topographie et Structure du Payload JSON
- **Chemin racine des données :** `data -> bikes` (liste d'objets représentant les véhicules en libre-service disponibles).
- **Exhaustivité des attributs bruts :**
  * `bike_id` (Identifiant unique/éphémère du véhicule)
  * `lat` / `lon` (Position géographique instantanée WGS84)
  * `is_reserved` / `is_disabled` (Statuts de disponibilité opérationnelle)
  * `current_fuel_percent` (Niveau de charge/carburant exprimé entre 0 et 1)
  * `current_range_meters` (Autonomie restante estimée en mètres)
  * `vehicle_type_id` (Identifiant du modèle de véhicule)

#### 💡 Spécificités d'Ingestion et Implications Analytiques :
- **Modèle Free-Floating vs Stations :** Absences de stations d'ancrage fixes. Les coordonnées `lat`/`lon` qualifient la position ponctuelle du véhicule à l'instant de la requête HTTP.
- **Normalisation des terminologies :** Bien que la norme GBFS utilise la nomenclature `bikes`, la source qualifie une flotte de **voitures en autopartage**. Une réécriture de la structure cible ou un alias de colonne (`vehicle_id`) sera mis en place en couche Silver.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** Requête HTTP GET vers le flux temps réel.
- **Stockage cible :** Table Delta `bronze_leogo_vehicles` en mode `overwrite` pour capturer la photo instantanée du parc disponible.


In [14]:
# 1. Requête HTTP GET vers Leo & Go avec application du timeout
url_leogo_lyon = "https://download.data.grandlyon.com/files/rdata/lag_leoandgo.disponibilite/free_bike_status.json"
response = requests.get(url_leogo_lyon, timeout=TIMEOUT_REQUETE)

# 2. Vérification du succès de la connexion
if response.status_code == 200:
    # 2.1 Conversion de la réponse brute en dictionnaire JSON
    donnees_json = response.json()
    
    # 2.2 Extraction ciblée de la liste des vélos à l'intérieur de la clé 'data -> bikes'
    leogo_lyon_list = donnees_json.get("data", {}).get("bikes", [])
    
    # 2.3 Création du DataFrame Pandas brut (Couche Bronze)
    df_leogo_lyon = pd.DataFrame(leogo_lyon_list)
    
    # 2.4 Ajout des colonnes de traçabilité (Règle d'or de la couche Bronze)
    df_leogo_lyon["_source_url"] = url_leogo_lyon
    df_leogo_lyon["_source_last_updated"] = donnees_json.get("last_updated")
    df_leogo_lyon["_ingestion_ts"] = pd.Timestamp.utcnow()

    # 2.5 Contrôle qualité demandé par l'audit (détection des coordonnées à 0.0)
    nb_zero = (df_leogo_lyon['lat'] == 0).sum() + (df_leogo_lyon['lon'] == 0).sum()
    
    # 2.6 Affichage du succès (Correction de la faute "succcès")
    print(f"🎉 Succès ! {len(df_leogo_lyon)} véhicules Leo & Go importés avec succès (Couche Bronze).")
    print(f"⚠️ Contrôle qualité : {nb_zero} coordonnées aberrantes à 0.0 détectées.")
    
    # 2.7 Aperçu visuel
    display(df_leogo_lyon[["bike_id", "last_reported", "is_reserved", "is_disabled", "lat", "lon", "current_fuel_percent"]].head())

else:
    # 3. Gestion du cas d'erreur
    print(f"❌ Erreur lors du chargement ! Code retour : {response.status_code}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 22, Finished, Available, Finished, False)

🎉 Succès ! 301 véhicules Leo & Go importés avec succès (Couche Bronze).
⚠️ Contrôle qualité : 0 coordonnées aberrantes à 0.0 détectées.


SynapseWidget(Synapse.DataFrame, 66e5c532-e114-4a3a-b994-3f1dd10a6697)

### 🎉 Résultat de l'ingestion brute (Couche Bronze) - Leo & Go

* **Succès du chargement :** Les véhicules Leo & Go ont été extraits avec succès et convertis en DataFrame Pandas brut.
* **Contrôle qualité :** Le code a permis de vérifier programmatiquement la présence (ou l'absence) de coordonnées aberrantes à `0.0` dans le flux.
* **Note pour la couche Silver :** Un géofiltrage strict sera nécessaire pour ne conserver que les véhicules situés dans la Métropole de Lyon.

### 🗺️ Validation Spatiale : Cartographie interactive des stations Leo & Go

Avant de procéder à la persistance définitive des données dans la couche Bronze, nous effectuons une validation visuelle et spatiale via une carte interactive.

* **Fond de carte :** Utilisation d'un fond Esri neutre (`World Light Gray Base`) pour mettre en valeur les données sans interférence visuelle excessive.
* **Représentation graphique :** Chaque station géolocalisée est matérialisée par un marqueur circulaire (`CircleMarker`).
* **Interactivité :** Une infobulle (popup HTML) au clic permet de consulter instantanément les attributs clés de la station (`nom`, `ID`, `capacité`).

In [15]:
# 1. Initialisation de la carte interactive centrée sur le cœur de Lyon avec un fond Esri neutre
carte_lyon_leogo = folium.Map(
    location=[45.7640, 4.8357],  # Point central de Lyon : [Latitude, Longitude]
    zoom_start=13,               # Niveau de zoom initial pour englober la zone urbaine
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", # URL du serveur de tuiles géographiques neutres
    attr="Esri"                  # Mention légale obligatoire (attribution du fournisseur de la carte)
)

# 2. Parcours itératif ligne par ligne du DataFrame Pandas (utilisation de 'station' pour plus de lisibilité)
for index, station in df_leogo_lyon.iterrows():
    
    # 3. Contrôle d'intégrité : on vérifie impérativement que les coordonnées GPS existent et ne sont pas vides (NaN)
    if pd.notna(station['lat']) and pd.notna(station['lon']):
        
        # 4. Structuration du contenu HTML de l'infobulle (popup) sur plusieurs lignes grâce aux triple guillemets
        info_bulle = f"""
        <b>Station :</b> {station.get('name', 'Inconnue')}<br>
        <b>ID :</b> {station.get('station_id', 'N/A')}<br>
        <b>Capacité :</b> {station.get('capacity', 'N/A')} véhicules
        """
        
        # 5. Création et ajout d'un marqueur circulaire stylisé pour chaque station valide
        folium.CircleMarker(
            location=[station['lat'], station['lon']], # Positionnement précis de la station
            radius=6,                                  # Taille du cercle en pixels
            color='#2b5c8f',                           # Couleur de la bordure du cercle
            weight=1,                                  # Épaisseur de la bordure
            fill=True,                                 # Active le remplissage de la forme
            fill_color='#3182ce',                      # Couleur de remplissage intérieure
            fill_opacity=0.8,                          # Transparence du remplissage
            popup=folium.Popup(info_bulle, max_width=300) # Intégration du bloc HTML interactif au clic
        ).add_to(carte_lyon_leogo)                     # Ajout de ce marqueur sur notre objet carte global

# 6. Affichage final de la carte interactive dans l'environnement Microsoft Fabric / Jupyter
display(carte_lyon_leogo)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 23, Finished, Available, Finished, False)

### 💾 Persistance des données brutes dans le Lakehouse (Couche Bronze)

Après validation spatiale sur la carte interactive, nous procédons à la sauvegarde des données brutes dans la table Delta de la couche Bronze.

* **Format de stockage :** Delta Lake (persistance orientée colonnes, ACID et versionnée).
* **Mode d'écriture :** `overwrite` (remplacement complet à chaque cycle d'ingestion brute).
* **Règle métier :** Les données enregistrées sont strictement identiques aux données brutes renvoyées par l'API Dott, sans aucune transformation.

In [16]:
# Persistance des données brutes Leo & Go dans le Lakehouse (Couche Bronze)
table_name_leogo = "bronze_leogo_vehicules"

spark_df_leogo = spark.createDataFrame(df_leogo_lyon)

(
    spark_df_leogo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name_leogo)
)

print(f"✅ Table Delta '{table_name_leogo}' enregistrée avec succès !")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 24, Finished, Available, Finished, False)

✅ Table Delta 'bronze_leogo_vehicules' enregistrée avec succès !


## Partie 3 : Ingestion et Visualisation des flux de transports en commun (Couche Bronze)

Dans cette partie, nous procédons à l'ingestion des données relatives aux **transports en commun à Lyon**. 

### Pourquoi dissocier les transports en commun de la micromobilité ?
Au départ, nous aurions pu supposer que les transports en commun et la micromobilité partageaient des dynamiques proches. Cependant, l'analyse montre que leurs flux, leur structure de données et leur utilisation au quotidien par les usagers sont fondamentalement différents. C'est pourquoi nous les traitons séparément. De plus, pour appréhender correctement l'offre de transport en commun, nous ne nous basons pas sur un flux unique, mais sur **plusieurs flux de données complémentaires**.

**Objectifs de cette étape :**
1. **Chargement des flux bruts :** Ingestion de l'ensemble des flux de données des transports en commun dans la couche Bronze, **sans aucun croisement de données ni analyse** à ce stade.
2. **Inspection visuelle et structurelle :** Affichage des en-têtes et des premières lignes pour analyser en détail la structure brute de chaque flux.
3. **Recommandations pour la couche Silver :** Établir des pistes et des recommandations concrètes basées sur cette inspection structurelle en vue du prochain nettoyage.


### 🔍 Analyse et Spécifications de la Source : Points d'arrêt TCL (`tcl_sytral.tclarret`)

- **Fournisseur / Opérateur :** SYTRAL Mobilités / Métropole de Lyon (Grand Lyon)
- **URL de l'API / Flux (paginée) :** `https://data.grandlyon.com/geoserver/sytral/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=sytral:tcl_sytral.tclarret&outputFormat=application/json&SRSNAME=EPSG:4326&startIndex=0&count=2000&sortby=gid`
- **Format d'entrée :** GeoJSON (OGC WFS 2.0.0)
- **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et l'affichage Folium
- **Indicateurs de fraîcheur (métadonnées du flux) :**
  * `timeStamp` (racine de la réponse) : horodatage de la requête côté serveur GeoServer
  * `numberMatched` / `numberReturned` : volumétrie totale attendue / retournée par page
  * `properties.last_update` / `properties.last_update_fme` : horodatage de mise à jour de chaque entité (source et traitement FME)

#### 🌳 Topographie et Structure du Payload JSON
- **Chemin racine des données :** `features` (liste d'objets GeoJSON `Feature` représentant chaque point d'arrêt physique)
- **Métadonnées globales de la collection :** `timeStamp`, `numberMatched`, `numberReturned`
- **Exhaustivité des attributs bruts (dans `properties` et `geometry`) :**
  * `id` (identifiant unique, ex. `tcl_sytral.tclarret.1`)
  * `nom` (libellé de l'arrêt, ex. `24 Août`, `Pont de l'Université`)
  * `desserte` (lignes desservant l'arrêt, ex. `145:A,87:A`)
  * `pmr`, `ascenseur`, `escalator` (booléens d'accessibilité)
  * `gid` (identifiant technique interne)
  * `adresse`, `commune`, `insee`, `zone`
  * `geometry` (Point en `EPSG:4326` : `[lon, lat]` en degrés)

#### ⚠️ Points de Vigilance Techniques (Couche Silver) :
- **Champ `desserte` :** chaîne concaténée (ex. `115:A,118:A,JD767:R`) — parsing nécessaire en Silver pour des filtres par ligne.
- **Pagination obligatoire :** GeoServer tronque la réponse à ~1 000 entités par défaut ; le réseau TCL dépasse ce seuil. L'ingestion doit boucler sur `startIndex` par pages de 2 000 jusqu'à `numberReturned < count`.
- **Normalisation temporelle :** conversion des champs ISO `last_update` / `last_update_fme` en `datetime` natif PySpark.
- **Extraction géométrique :** coordonnées imbriquées dans `geometry.coordinates` ; extraction à prévoir en Silver.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requêtes WFS paginées via `requests.get(timeout=TIMEOUT_REQUETE)`, avec `raise_for_status()` bloquant et retry borné.
- **Principe :** zéro manipulation locale ; données brutes persistées telles quelles dans `bronze_tcl_arrets` (Delta), avec ajout des colonnes techniques `_source_url`, `_source_last_updated`, `_source_ttl`, `_source_version`, `_ingestion_ts`.


In [17]:
# 1. Requête HTTP GET paginée vers le flux WFS des arrêts TCL (EPSG:4326, timeout, retry borné)
import time

url_tcl_base = (
    "https://data.grandlyon.com/geoserver/sytral/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=sytral:tcl_sytral.tclarret"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&sortby=gid"
)
PAGE_SIZE = 2000
MAX_RETRIES = 3

tous_les_arrets = []
start_index = 0
derniere_meta = {}

print("⏳ Extraction paginée des arrêts TCL en cours...")

while True:
    url_paginee = f"{url_tcl_base}&startIndex={start_index}&count={PAGE_SIZE}"

    # Retry borné pour absorber les timeouts réseau ponctuels
    for tentative in range(MAX_RETRIES):
        try:
            response = requests.get(url_paginee, timeout=TIMEOUT_REQUETE)
            response.raise_for_status()
            break
        except requests.exceptions.Timeout:
            print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES}) à startIndex={start_index}")
            time.sleep(2)
    else:
        raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives à startIndex={start_index}")

    donnees_json = response.json()
    derniere_meta = donnees_json  # on garde la dernière réponse pour les métadonnées
    features = donnees_json.get("features", [])

    if not features:
        break

    tous_les_arrets.extend(features)
    print(f"📥 {len(features)} arrêts récupérés (cumul : {len(tous_les_arrets)})")

    # Sortie propre : moins de lignes que demandé = dernière page
    if len(features) < PAGE_SIZE:
        break

    start_index += PAGE_SIZE

# 2. Création du DataFrame Pandas brut (Couche Bronze)
df_tcl_arrets = pd.DataFrame(tous_les_arrets)

# 3. Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
df_tcl_arrets["_source_url"]          = url_tcl_base
df_tcl_arrets["_source_last_updated"] = derniere_meta.get("timeStamp")
df_tcl_arrets["_source_ttl"]          = None                    # pas de TTL explicite côté WFS
df_tcl_arrets["_source_version"]      = derniere_meta.get("numberMatched")
df_tcl_arrets["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

# 4. Bilan d'ingestion
print(f"🎉 Succès ! {len(df_tcl_arrets)} arrêts TCL ingérés depuis la source complète.")
print(f"   Metadata flux → timeStamp={derniere_meta.get('timeStamp')} | numberMatched={derniere_meta.get('numberMatched')}")

# 5. Aperçu visuel
display(df_tcl_arrets.head())

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 25, Finished, Available, Finished, True)

⏳ Extraction paginée des arrêts TCL en cours...
📥 2000 arrêts récupérés (cumul : 2000)
📥 2000 arrêts récupérés (cumul : 4000)
📥 2000 arrêts récupérés (cumul : 6000)
📥 2000 arrêts récupérés (cumul : 8000)
📥 1766 arrêts récupérés (cumul : 9766)
🎉 Succès ! 9766 arrêts TCL ingérés depuis la source complète.
   Metadata flux → timeStamp=2026-11-11T00:04:55.390Z | numberMatched=9766


SynapseWidget(Synapse.DataFrame, 817581d6-7959-4fe0-ab2b-61331caf12a5)

In [18]:
from folium.plugins import FastMarkerCluster

# 1. Initialisation de la carte interactive centrée sur Lyon (fond Esri neutre)
carte_lyon_tcl = folium.Map(
    location=[45.7640, 4.8357],
    zoom_start=13,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Extraction des coordonnées et des libellés pour le cluster
coordonnees_tcl = []
nb_ignorees = 0

for _, arret in df_tcl_arrets.iterrows():
    geom = arret.get("geometry")
    props = arret.get("properties")

    # Sécurisation : geometry doit être un dict avec coordinates valides
    if not isinstance(geom, dict):
        nb_ignorees += 1
        continue
    coords = geom.get("coordinates")
    if not coords or len(coords) < 2:
        nb_ignorees += 1
        continue

    lon, lat = coords[0], coords[1]
    if pd.isna(lat) or pd.isna(lon):
        nb_ignorees += 1
        continue

    # Sécurisation : properties doit être un dict
    if not isinstance(props, dict):
        props = {}

    nom_arret  = props.get("nom", "Inconnu")
    desserte   = props.get("desserte", "N/C")
    popup_text = f"Arrêt : {nom_arret} | Lignes : {desserte}"

    coordonnees_tcl.append([lat, lon, popup_text])

# 3. Ajout via FastMarkerCluster pour absorber les ~4 000 points sans saturer le notebook
FastMarkerCluster(data=coordonnees_tcl).add_to(carte_lyon_tcl)

# 4. Bilan et affichage
print(f"✅ {len(coordonnees_tcl)} arrêts affichés sur la carte ({nb_ignorees} lignes ignorées faute de géométrie valide).")
display(carte_lyon_tcl)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 26, Submitted, Running, Running, True)

✅ 9766 arrêts affichés sur la carte (0 lignes ignorées faute de géométrie valide).


In [ ]:
# 1. Nom de table cible (convention bronze_<source>_<objet>)
table_name = "bronze_tcl_arrets"

# 2. Conversion du DataFrame Pandas en DataFrame Spark
spark_df_tcl_arrets = spark.createDataFrame(df_tcl_arrets)

# 3. Écriture Delta en mode overwrite (Bronze : réingestion complète à chaque run)
(
    spark_df_tcl_arrets.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # autorise l'évolution du schéma (utile si SYTRAL ajoute un champ)
    .saveAsTable(table_name)
)

# 4. Bilan
print(f"✅ Table Delta '{table_name}' enregistrée ({len(df_tcl_arrets)} lignes).")

StatementMeta(, , -1, Waiting, , Waiting, True)

## 🔍 Analyse et Spécifications de la Source : Parcs Relais TCL (`tcl_sytral.tclparcrelaisst`)

- **Fournisseur / Opérateur :** SYTRAL / Métropole de Lyon
- **URL de l'API (paginée) :** `https://data.grandlyon.com/geoserver/sytral/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=sytral:tcl_sytral.tclparcrelaisst&outputFormat=application/json&SRSNAME=EPSG:4326&startIndex=0&count=5000&sortby=gid`
- **Format d'entrée :** GeoJSON (OGC WFS 2.0.0)
- **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium
- **Indicateurs de fraîcheur (métadonnées du flux) :**
  * `timeStamp` (racine) : horodatage de la requête côté GeoServer
  * `numberMatched` / `numberReturned` : volumétrie attendue / retournée
  * `properties.last_update` / `properties.last_update_fme` : horodatage de mise à jour par entité

### 1. Structure globale du dictionnaire JSON
- **Racine (`FeatureCollection`) :** collection standard de caractéristiques géographiques GeoJSON.
- **Clé principale (`features`) :** liste d'objets, chaque objet représentant un parc relais individuel du réseau TCL.

### 2. Dictionnaires imbriqués par entité (`Feature`)
Chaque objet de `features` possède quatre clés principales :
- **`type` :** `"Feature"`.
- **`id` :** identifiant unique global côté serveur WFS (ex. `"tcl_sytral.tclparcrelaisst.ALP"`) — distinct du code métier court dans `properties.id`.
- **`geometry` :** objet géographique :
  * `type` : `"MultiPoint"`
  * `coordinates` : tableau imbriqué `[[lon, lat]]` en `EPSG:4326`. Extraction en Silver via `coords[0][0]` / `coords[0][1]`.
- **`properties` :** dictionnaire métier :
  * `"id"` : code court du parc (ex. `"ALP"`).
  * `"nom"` : libellé officiel (ex. `"Porte des Alpes"`).
  * `"capacite"` : nombre total de places.
  * `"place_handi"` : nombre de places PMR.
  * `"horaires"` : plage d'ouverture détaillée (texte libre).
  * `"p_surv"` : indicateur booléen de surveillance.
  * `"gid"` : identifiant technique interne.
  * `"last_update"` / `"last_update_fme"` : horodatages.
- **`bbox` :** boîte englobante de l'entité.

---

### 💡 Intérêt Stratégique pour *Cities Mobility*
Ce flux fournit des métriques quantitatives cruciales (**capacité globale**, **places PMR**, **horaires d'affluence potentiels**) pour mesurer la capacité d'absorption des pôles d'échanges multimodaux où les automobilistes cherchent à basculer vers des mobilités douces ou partagées.

---

### ⚠️ Points de Vigilance Techniques et Métier
1. **La géométrie de type `MultiPoint` :**
   * Contrairement aux arrêts (`Point`) où `coordinates` est un tableau simple `[lon, lat]`, `MultiPoint` renvoie `[[lon, lat]]`. Il faut impérativement cibler `coords[0][0]` (lon) et `coords[0][1]` (lat) sous peine d'erreur de type en Silver ou d'absence d'affichage Folium.
2. **Hétérogénéité des règles d'accès (`horaires`) :**
   * Certains parcs imposent des restrictions strictes (`"Réservé aux abonnés TCL"`, fermetures dominicales). Traitement à prévoir en Silver pour filtrer les parcs réellement exploitables en usage autopartage.
3. **Fraîcheur des données :**
   * Suivi de `last_update` indispensable pour s'assurer que les capacités reflètent la réalité terrain avant tout croisement avec des sources temps réel (micromobilité).


In [ ]:
# 1. Requête HTTP GET vers le flux WFS des Parcs Relais (EPSG:4326, timeout, retry borné)
import time

url_parcs_relais = (
    "https://data.grandlyon.com/geoserver/sytral/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=sytral:tcl_sytral.tclparcrelaisst"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_parcs_relais, timeout=TIMEOUT_REQUETE)
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_parcs_relais}")

# 2. Conversion de la réponse brute en dictionnaire Python
donnees_json = response.json()

# 3. Extraction brute de la liste des entités (features)
parcs_relais_list = donnees_json.get("features", [])

# 4. Création du DataFrame Pandas brut (Couche Bronze)
df_parcs_relais = pd.DataFrame(parcs_relais_list)

# 5. Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
df_parcs_relais["_source_url"]          = url_parcs_relais
df_parcs_relais["_source_last_updated"] = donnees_json.get("timeStamp")
df_parcs_relais["_source_ttl"]          = None               # pas de TTL explicite côté WFS
df_parcs_relais["_source_version"]      = donnees_json.get("numberMatched")
df_parcs_relais["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

# 6. Bilan d'ingestion
print(f"🎉 Succès ! {len(df_parcs_relais)} parcs relais TCL chargés à l'état brut (Couche Bronze).")
print(f"   Metadata flux → timeStamp={donnees_json.get('timeStamp')} | numberMatched={donnees_json.get('numberMatched')}")

# 7. Aperçu brut
display(df_parcs_relais.head())

StatementMeta(, , -1, Waiting, , Waiting, True)

### 📝 Résultat de l'ingestion Bronze : Parcs Relais TCL

* **Statut :** 🎉 Succès ! 22 parcs relais TCL chargés à l'état brut.
* **Observation technique :** Le DataFrame présente de nombreuses données imbriquées (notamment les dictionnaires `geometry` de type MultiPoint et `properties`). Ces structures complexes nécessiteront une attention particulière et un traitement de désimbrication rigoureux lors du passage à la couche **Silver**.

### 🗺️ Visualisation cartographique : Parcs Relais TCL

* **Objectif :** Afficher l'ensemble des parcs relais sur une carte interactive centrée sur Lyon avec un fond neutre (Esri), en extrayant proprement les coordonnées depuis la structure `MultiPoint`.

In [ ]:
# 1. Initialisation de la carte interactive centrée sur Lyon (fond Esri neutre)
carte_lyon_parcs = folium.Map(
    location=[45.7640, 4.8357],
    zoom_start=13,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Boucle d'extraction et d'ajout des parcs relais (gestion du MultiPoint)
compteur = 0
nb_ignorees = 0

for _, row in df_parcs_relais.iterrows():
    geom  = row.get("geometry")
    props = row.get("properties")

    # Sécurisation : geometry doit être un dict avec coordinates valides
    if not isinstance(geom, dict):
        nb_ignorees += 1
        continue
    coords = geom.get("coordinates", [])
    if not coords or not isinstance(coords[0], list) or len(coords[0]) < 2:
        nb_ignorees += 1
        continue

    # MultiPoint → coords[0] = [lon, lat] en EPSG:4326
    lon, lat = coords[0][0], coords[0][1]
    if pd.isna(lat) or pd.isna(lon):
        nb_ignorees += 1
        continue

    # Sécurisation : properties doit être un dict
    if not isinstance(props, dict):
        props = {}

    popup_text = (
        f"<b>Parc Relais :</b> {props.get('nom','N/C')}<br>"
        f"<b>Capacité :</b> {props.get('capacite','N/C')} places<br>"
        f"<b>Horaires :</b> {props.get('horaires','N/C')}"
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color="#2b5c8f",
        fill=True,
        fill_color="#3498db",
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(carte_lyon_parcs)
    compteur += 1

# 3. Bilan et affichage
print(f"✅ {compteur} parcs relais affichés sur la carte ({nb_ignorees} lignes ignorées faute de géométrie valide).")
display(carte_lyon_parcs)

StatementMeta(, , -1, Waiting, , Waiting, True)

### 💾 Persistance Lakehouse (Couche Bronze) : Parcs Relais TCL

* **Objectif :** Sauvegarder le DataFrame brut des parcs relais TCL dans le catalogue Lakehouse sous forme de table Delta (`bronze_tcl_parcs_relais`), en appliquant l'option d'écrasement de schéma pour anticiper les structures imbriquées.

In [ ]:
# 1. Nom de table cible (convention bronze_<source>_<objet>)
table_name = "bronze_tcl_parcs_relais"

# 2. Conversion du DataFrame Pandas en DataFrame Spark
spark_df_tcl_parcs_relais = spark.createDataFrame(df_parcs_relais)

# 3. Écriture Delta en mode overwrite (Bronze : réingestion complète à chaque run)
(
    spark_df_tcl_parcs_relais.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")  # autorise l'évolution du schéma (utile si SYTRAL ajoute un champ)
    .saveAsTable(table_name)
)

# 4. Bilan
print(f"✅ Table Delta '{table_name}' enregistrée ({len(df_parcs_relais)} lignes).")

StatementMeta(, , -1, Waiting, , Waiting, True)

## 🔍 Analyse et Spécifications de la Source : Lignes de Métro et Funiculaire TCL (`tcl_sytral.tcllignemf_2_0_0`)

* **Fournisseur / Opérateur :** SYTRAL / Métropole de Lyon
* **URL de l'API (paginée) :** `https://data.grandlyon.com/geoserver/sytral/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=sytral:tcl_sytral.tcllignemf_2_0_0&outputFormat=application/json&SRSNAME=EPSG:4326&startIndex=0&count=5000&sortby=gid`
* **Format d'entrée :** GeoJSON (OGC WFS 2.0.0)
* **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium
* **Indicateurs de fraîcheur (métadonnées du flux) :**
  * `timeStamp` (racine) : horodatage de la requête côté GeoServer
  * `numberMatched` / `numberReturned` : volumétrie attendue / retournée
  * `properties.last_update` / `properties.nom_version` : version réseau et date de mise à jour par entité

### 1. Structure globale du dictionnaire JSON
- **Racine (`FeatureCollection`) :** collection standard de caractéristiques géographiques GeoJSON représentant le tracé des lignes lourdes (métro + funiculaire).
- **Clé principale (`features`) :** liste d'objets, chaque objet correspondant à un tronçon (ou un sens) d'une ligne de métro/funiculaire.
- **Volumétrie attendue :** ordre de grandeur d'une dizaine de tronçons (réseau métro + 2 funiculaires), très en dessous du seuil de troncature GeoServer.

### 2. Dictionnaires imbriqués par entité (`Feature`)
Chaque objet de `features` possède quatre clés principales :
- **`type` :** `"Feature"`.
- **`id` :** identifiant unique global côté serveur WFS (ex. `"tcl_sytral.tcllignemf_2_0_0.1"`).
- **`geometry` :** objet géographique linéaire :
  * `type` : `"MultiLineString"`
  * `coordinates` : tableau imbriqué `[[[lon, lat], [lon, lat], ...]]` en `EPSG:4326`.
- **`properties` :** dictionnaire métier :
  * `"gid"` : identifiant technique interne.
  * `"ligne"` / `"code_ligne"` : code de la ligne (ex. `"C"`, `"A"`).
  * `"code_trace"` / `"type_trace"` : identification du tracé (ex. `"C-1"`, `"NOM"` pour nominal).
  * `"nom_trace"` : libellé explicite du parcours (ex. `"Cuire - Hôtel de Ville Louis Pradel"`).
  * `"sens"` : direction du parcours (`"Aller"` ou `"Retour"`).
  * `"origine"` / `"destination"` et `"nom_origine"` / `"nom_destination"` : arrêts de début et de fin du tronçon.
  * `"famille_transport"` : code de la famille (ex. `"MET"` pour Métro).
  * `"couleur"` / `"couleur_hex"` : code couleur officiel de la ligne (ex. `#EC6608`).
  * `"pmr"` : indicateur d'accessibilité PMR (`true` / `false`).
  * `"nom_version"` : version du réseau (ex. `"Semaine 09 - 2026 V2"`).
  * `"last_update"` : horodatage de mise à jour des données.
- **`bbox` :** boîte englobante géographique de l'entité.

---

### 💡 Intérêt Stratégique pour *Cities Mobility*
Ce flux cartographie **l'ossature lourde et structurante** du réseau de transport en commun. Il est indispensable pour analyser la complémentarité entre l'offre de métro/funiculaire à fort débit et l'implantation de nos stations de véhicules partagés, permettant d'identifier les axes majeurs de rabattement ou de concurrence potentielle.

---

### ⚠️ Points de Vigilance Techniques et Métier
1. **La géométrie de type `MultiLineString` :**
   * Contrairement aux points (`Point` / `MultiPoint`), il s'agit ici de lignes composées d'une suite de coordonnées `[[[lon, lat], [lon, lat], ...]]`. L'exploitation cartographique nécessite `folium.PolyLine` avec inversion `[lon, lat] → [lat, lon]` à chaque point.
2. **Multiplicité des sens et des tronçons :**
   * Une même ligne de métro est découpée en plusieurs enregistrements selon le sens (`Aller` / `Retour`) et les éventuelles branches. Un travail de regroupement sera nécessaire en couche Silver.
3. **Champ `couleur_hex` :**
   * Valeur attendue au format `#RRGGBB`. Contrôle défensif à prévoir côté visualisation (`startswith('#')` + fallback) pour ne pas casser le rendu en cas de valeur vide ou mal typée.
4. **Fraîcheur des données :**
   * Suivi de `last_update` et `nom_version` indispensable pour tracer la version réseau utilisée lors de chaque exécution du pipeline.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requête WFS unique (`count=5000`, pas de pagination nécessaire vu le volume), avec `requests.get(timeout=TIMEOUT_REQUETE)`, `raise_for_status()` bloquant et retry borné.
- **Principe :** zéro manipulation locale ; données brutes persistées telles quelles dans `bronze_tcl_lignes_mf` (Delta), avec ajout des colonnes techniques `_source_url`, `_source_last_updated`, `_source_ttl`, `_source_version`, `_ingestion_ts`.


In [ ]:
# 1. Requête HTTP GET vers le flux WFS des Lignes Métro / Funiculaire (EPSG:4326, timeout, retry borné)
import time

url_lignes_mf = (
    "https://data.grandlyon.com/geoserver/sytral/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=sytral:tcl_sytral.tcllignemf_2_0_0"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_lignes_mf, timeout=TIMEOUT_REQUETE)
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_lignes_mf}")

# 2. Conversion de la réponse brute en dictionnaire Python
donnees_json = response.json()

# 3. Extraction brute de la liste des entités (features)
lignes_mf_list = donnees_json.get("features", [])

# 4. Création du DataFrame Pandas brut (Couche Bronze)
df_lignes_mf = pd.DataFrame(lignes_mf_list)

# 5. Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
df_lignes_mf["_source_url"]          = url_lignes_mf
df_lignes_mf["_source_last_updated"] = donnees_json.get("timeStamp")
df_lignes_mf["_source_ttl"]          = None               # pas de TTL explicite côté WFS
df_lignes_mf["_source_version"]      = donnees_json.get("numberMatched")
df_lignes_mf["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

# 6. Bilan d'ingestion
print(f"🎉 Succès ! {len(df_lignes_mf)} tronçons de lignes de métro et funiculaire TCL chargés à l'état brut (Couche Bronze).")
print(f"   Metadata flux → timeStamp={donnees_json.get('timeStamp')} | numberMatched={donnees_json.get('numberMatched')}")

# 7. Aperçu brut
display(df_lignes_mf.head())

StatementMeta(, , -1, Waiting, , Waiting, True)

### 📝 Résultat de l'ingestion Bronze : Lignes de Métro et Funiculaire TCL

* **Statut :** 🎉 Succès ! 12 tronçons de lignes de métro et funiculaire TCL chargés à l'état brut.
* **Colonnes clés méritant notre attention (Couche Silver) :**
  * `geometry` : Contient les tracés géographiques complexes au format `MultiLineString` qui nécessiteront un traitement géométrique spécifique pour l'affichage cartographique.
  * `properties` : Regroupe l'ensemble des métadonnées métiers imbriquées (code de la ligne, nom du tracé, sens de circulation, accessibilité PMR) qui devront être aplaties lors de la structuration de la couche Silver.

### 🗺️ Visualisation cartographique : Lignes de Métro et Funiculaire TCL

* **Objectif :** Tracer les lignes de métro et funiculaire sur une carte interactive en extrayant les coordonnées des structures `MultiLineString` et en appliquant les couleurs officielles de chaque ligne.

In [ ]:
# 1. Initialisation de la carte interactive centrée sur Lyon avec un fond Esri neutre
carte_lyon_lignes = folium.Map(
    location=[45.7640, 4.8357],    # Point central de Lyon : [Latitude, Longitude]
    zoom_start=13,                 # Niveau de zoom initial
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Boucle d'extraction et d'ajout des tracés linéaires sur la carte
for index, row in df_lignes_mf.iterrows():
    geom = row.get("geometry")
    props = row.get("properties")
    
    if geom and isinstance(geom, dict):
        coords = geom.get("coordinates", [])
        # Récupération des propriétés pour les popups et le style
        if props and isinstance(props, str):
            # Si properties est sous forme de chaîne JSON (selon l'ingestion brute)
            try:
                props_dict = json.loads(props)
            except:
                props_dict = {}
        elif isinstance(props, dict):
            props_dict = props
        else:
            props_dict = {}
            
        nom_ligne = props_dict.get("ligne", "Inconnue")
        nom_trace = props_dict.get("nom_trace", "Tracé")
        sens = props_dict.get("sens", "")
        
        # Récupération de la couleur hexadécimale officielle, avec une valeur par défaut si absente
        couleur_hex = props_dict.get("couleur_hex", "#3498db")
        if not couleur_hex or not couleur_hex.startswith("#"):
            couleur_hex = "#3498db"
            
        popup_text = f"<b>Ligne :</b> Métro {nom_ligne}<br><b>Parcours :</b> {nom_trace}<br><b>Sens :</b> {sens}"
        
        # Le format MultiLineString est une liste de lignes, chaque ligne étant une liste de points [lon, lat]
        for line_coords in coords:
            # Inversion de [lon, lat] en [lat, lon] pour Folium
            latlon_coords = [[pt[1], pt[0]] for pt in line_coords if len(pt) >= 2]
            
            if latlon_coords:
                folium.PolyLine(
                    locations=latlon_coords,
                    color=couleur_hex,
                    weight=5,
                    opacity=0.8,
                    popup=folium.Popup(popup_text, max_width=300)
                ).add_to(carte_lyon_lignes)

# 3. Affichage final de la carte
display(carte_lyon_lignes)

StatementMeta(, , -1, Waiting, , Waiting, True)

### 💾 Persistance Lakehouse (Couche Bronze) : Lignes de Métro et Funiculaire TCL

* **Objectif :** Sauvegarder le DataFrame brut des lignes de métro et funiculaire dans le catalogue Lakehouse sous forme de table Delta (`bronze_tcl_lignes_mf`), en appliquant l'option d'écrasement de schéma pour gérer les structures géographiques complexes.

In [25]:
# 1. Définition du nom de la table cible dans le Lakehouse avec le préfixe 'bronze_'
table_name = "bronze_tcl_lignes_mf"

try:
    # 2. Conversion sécurisée du DataFrame Pandas en DataFrame Spark
    spark_df_lignes = spark.createDataFrame(df_lignes_mf)

    # 3. Écriture de la table persistante dans le Lakehouse
    (
        spark_df_lignes.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"✅ Succès ! Table Delta '{table_name}' enregistrée avec succès dans la couche Bronze du Lakehouse ({len(df_lignes_mf)} lignes traitées).")

except Exception as e:
    print(f"❌ Erreur lors de l'enregistrement Spark : {str(e)}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 33, Finished, Available, Finished, True)

✅ Succès ! Table Delta 'bronze_tcl_lignes_mf' enregistrée avec succès dans la couche Bronze du Lakehouse (12 lignes traitées).


## 🔍 Analyse et Spécifications de la Source : Lignes de Tramway TCL (`tcl_sytral.tcllignetram_2_0_0`)

* **Fournisseur / Opérateur :** SYTRAL / Métropole de Lyon
* **URL de l'API WFS :** `https://data.grandlyon.com/geoserver/sytral/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=sytral:tcl_sytral.tcllignetram_2_0_0&outputFormat=application/json&SRSNAME=EPSG:4326&startIndex=0&count=5000&sortby=gid`
* **Format d'entrée :** GeoJSON (OGC WFS 2.0.0)
* **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium
* **Indicateurs de fraîcheur (métadonnées du flux) :**
  * `timeStamp` (racine) : horodatage de la requête côté GeoServer
  * `numberMatched` / `numberReturned` : volumétrie attendue / retournée
  * `properties.last_update` : horodatage de mise à jour par entité

### 1. Structure globale du dictionnaire JSON
- **Racine (`FeatureCollection`) :** collection de caractéristiques géographiques représentant l'ensemble des tronçons de tramway.
- **Clé principale (`features`) :** liste d'entités géographiques contenant les tracés et les attributs métiers de chaque ligne (T1, T2, T3, T4, T5, T6, T7, T8, etc.).
- **Volumétrie attendue :** ordre de grandeur d'une vingtaine de tronçons, largement sous le seuil de troncature GeoServer avec `count=5000` → **pas de boucle paginée nécessaire**.

### 2. Dictionnaires imbriqués par entité (`Feature`)
- **`type` :** `"Feature"`.
- **`id` :** identifiant unique global côté serveur WFS.
- **`geometry` :** objet géographique linéaire :
  * `type` : `"MultiLineString"`
  * `coordinates` : tableau imbriqué `[[[lon, lat], [lon, lat], ...]]` en `EPSG:4326`.
- **`properties` :** métadonnées associées incluant :
  * `"ligne"` / `"code_ligne"` : identifiant commercial de la ligne (ex. `"T1"`, `"T2"`).
  * `"nom_trace"` : libellé du parcours (ex. `"Debourg - IUT Feyssine"`).
  * `"sens"` : sens de circulation (`"Aller"` / `"Retour"`).
  * `"couleur_hex"` : code couleur officiel pour le rendu cartographique.
  * `"pmr"` : accessibilité PMR (`true` / `false`).
  * `"last_update"` : horodatage de mise à jour.

---

### 💡 Intérêt Stratégique pour *Cities Mobility*
Le tramway irrigue les zones intermédiaires et périphériques de la métropole. Il représente un levier d'intermodalité crucial pour positionner nos stations de mobilité douce et partagée en synergie directe avec les stations de tramway à fort flux pendulaire.

---

### ⚠️ Points de Vigilance Techniques et Métier
1. **La géométrie de type `MultiLineString` :**
   * Suite de coordonnées `[[[lon, lat], ...]]` → `folium.PolyLine` avec inversion `[lon, lat] → [lat, lon]` à chaque point.
2. **Multiplicité des sens et des tronçons :**
   * Une même ligne de tramway est découpée en plusieurs enregistrements selon le sens (`Aller` / `Retour`) et les éventuelles branches. Un travail de regroupement sera nécessaire en couche Silver.
3. **Champ `couleur_hex` :**
   * Contrôle défensif à prévoir côté visualisation (`startswith('#')` + fallback `#e67e22`) pour ne pas casser le rendu en cas de valeur vide ou mal typée.
4. **Fraîcheur des données :**
   * Suivi de `last_update` indispensable pour tracer la version réseau utilisée lors de chaque exécution.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requête WFS unique (`count=5000`, pas de pagination nécessaire vu la volumétrie), avec `requests.get(timeout=TIMEOUT_REQUETE)`, `raise_for_status()` bloquant et retry borné.
- **Principe :** zéro manipulation locale ; données brutes persistées telles quelles dans `bronze_tcl_lignes_tram` (Delta), avec ajout des colonnes techniques `_source_url`, `_source_last_updated`, `_source_ttl`, `_source_version`, `_ingestion_ts`.


In [26]:
# 1. Requête HTTP GET vers le flux WFS des Lignes de Tramway (EPSG:4326, timeout, retry borné)
import time

url_lignes_tram = (
    "https://data.grandlyon.com/geoserver/sytral/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=sytral:tcl_sytral.tcllignetram_2_0_0"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_lignes_tram, timeout=TIMEOUT_REQUETE)
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_lignes_tram}")

# 2. Conversion de la réponse brute en dictionnaire Python
donnees_json = response.json()

# 3. Extraction brute de la liste des entités (features)
lignes_tram_list = donnees_json.get("features", [])

# 4. Création du DataFrame Pandas brut (Couche Bronze)
df_lignes_tram = pd.DataFrame(lignes_tram_list)

# 5. Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
df_lignes_tram["_source_url"]          = url_lignes_tram
df_lignes_tram["_source_last_updated"] = donnees_json.get("timeStamp")
df_lignes_tram["_source_ttl"]          = None               # pas de TTL explicite côté WFS
df_lignes_tram["_source_version"]      = donnees_json.get("numberMatched")
df_lignes_tram["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

# 6. Bilan d'ingestion
print(f"🎉 Succès ! {len(df_lignes_tram)} tronçons de lignes de tramway TCL chargés à l'état brut (Couche Bronze).")
print(f"   Metadata flux → timeStamp={donnees_json.get('timeStamp')} | numberMatched={donnees_json.get('numberMatched')}")

# 7. Aperçu brut
display(df_lignes_tram.head())

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 34, Finished, Available, Finished, True)

🎉 Succès ! 18 tronçons de lignes de tramway TCL chargés à l'état brut (Couche Bronze).
   Metadata flux → timeStamp=2026-11-11T00:06:01.729Z | numberMatched=18


SynapseWidget(Synapse.DataFrame, fd6b7fa4-22cb-465e-99fc-30546280f974)

### 📝 Résultat de l'ingestion Bronze : Lignes de Tramway TCL

* **Statut :** 🎉 Succès ! 18 tronçons de lignes de tramway TCL chargés à l'état brut.
* **Colonnes clés méritant notre attention (Couche Silver) :**
  * `geometry` : Contient les tracés linéaires complexes au format `MultiLineString` qu'il faudra traiter et convertir proprement pour l'affichage cartographique interactif.
  * `properties` : Regroupe l'ensemble des attributs métiers imbriqués (codes lignes, libellés de parcours, sens, codes couleurs officiels et accessibilité PMR) qui devront être complètement désimbriqués et structurés lors du passage à la couche Silver.

### 🗺️ Visualisation cartographique : Lignes de Tramway TCL

* **Objectif :** Tracer l'ensemble des lignes de tramway sur une carte interactive centrée sur Lyon en extrayant les coordonnées des structures `MultiLineString` et en appliquant les couleurs officielles de chaque ligne.

In [27]:
import folium

# 1. Initialisation de la carte interactive centrée sur Lyon avec un fond Esri neutre
carte_lyon_tram = folium.Map(
    location=[45.7640, 4.8357],    # Point central de Lyon : [Latitude, Longitude]
    zoom_start=13,                 # Niveau de zoom initial
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Boucle d'extraction et d'ajout des tracés de tramway sur la carte
for index, row in df_lignes_tram.iterrows():
    geom = row.get("geometry")
    props = row.get("properties")
    
    if geom and isinstance(geom, dict):
        coords = geom.get("coordinates", [])
        
        # Extraction sécurisée des propriétés métiers
        if isinstance(props, dict):
            props_dict = props
        else:
            props_dict = {}
            
        nom_ligne = props_dict.get("ligne", "Inconnue")
        nom_trace = props_dict.get("nom_trace", "Tracé")
        sens = props_dict.get("sens", "")
        
        # Récupération de la couleur hexadécimale officielle, avec une valeur par défaut si absente
        couleur_hex = props_dict.get("couleur_hex", "#e67e22")
        if not couleur_hex or not couleur_hex.startswith("#"):
            couleur_hex = "#e67e22"
            
        popup_text = f"<b>Ligne :</b> Tramway {nom_ligne}<br><b>Parcours :</b> {nom_trace}<br><b>Sens :</b> {sens}"
        
        # Le format MultiLineString est une liste de segments linéaires
        for line_coords in coords:
            # Inversion de [lon, lat] en [lat, lon] pour Folium
            latlon_coords = [[pt[1], pt[0]] for pt in line_coords if len(pt) >= 2]
            
            if latlon_coords:
                folium.PolyLine(
                    locations=latlon_coords,
                    color=couleur_hex,
                    weight=5,
                    opacity=0.8,
                    popup=folium.Popup(popup_text, max_width=300)
                ).add_to(carte_lyon_tram)

# 3. Affichage final de la carte
display(carte_lyon_tram)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 35, Finished, Available, Finished, True)

### 💾 Persistance Lakehouse (Couche Bronze) : Lignes de Tramway TCL

* **Objectif :** Sauvegarder le DataFrame brut des lignes de tramway dans le catalogue Lakehouse sous forme de table Delta (`bronze_tcl_lignes_tram`), en appliquant l'option d'écrasement de schéma.

In [28]:
# 1. Définition du nom de la table cible dans le Lakehouse avec le préfixe 'bronze_'
table_name = "bronze_tcl_lignes_tram"

try:
    # 2. Conversion du DataFrame Pandas en DataFrame Spark (moteur distribué)
    spark_df_tram = spark.createDataFrame(df_lignes_tram)

    # 3. Écriture de la table persistante dans le Lakehouse avec prise en compte du schéma
    (
        spark_df_tram.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # Permet de réinitialiser le schéma et d'éviter les erreurs de fusion des champs
        .saveAsTable(table_name)            # Enregistrement effectif de la table dans le catalogue du Lakehouse
    )

    # 4. Affichage d'un message de succès court et concis
    print(f"✅ Succès ! Table Delta '{table_name}' enregistrée avec succès dans la couche Bronze du Lakehouse ({len(df_lignes_tram)} lignes traitées).")

except Exception as e:
    print(f"❌ Erreur lors de l'enregistrement Spark : {str(e)}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 36, Finished, Available, Finished, False)

✅ Succès ! Table Delta 'bronze_tcl_lignes_tram' enregistrée avec succès dans la couche Bronze du Lakehouse (18 lignes traitées).


## 🔍 Analyse et Spécifications de la Source : Lignes de Bus TCL (`tcl_sytral.tcllignebus_2_0_0`)

* **Fournisseur / Opérateur :** SYTRAL / Métropole de Lyon
* **URL de l'API WFS :** `https://data.grandlyon.com/geoserver/sytral/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=sytral:tcl_sytral.tcllignebus_2_0_0&outputFormat=application/json&SRSNAME=EPSG:4326&startIndex=0&count=5000&sortby=gid`
* **Format d'entrée :** GeoJSON (OGC WFS 2.0.0)
* **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium
* **Indicateurs de fraîcheur (métadonnées du flux) :**
  * `timeStamp` (racine) : horodatage de la requête côté GeoServer
  * `numberMatched` / `numberReturned` : volumétrie attendue / retournée
  * `properties.last_update` : horodatage de mise à jour par entité

### 1. Structure globale du dictionnaire JSON
- **Racine (`FeatureCollection`) :** collection de caractéristiques géographiques recensant l'ensemble du réseau de bus de surface de la métropole.
- **Clé principale (`features`) :** liste d'entités contenant les géométries de type linéaire et les métadonnées de chaque ligne de bus (lignes majeures C, lignes régulières, lignes des lycées, navettes, etc.).
- **Volumétrie attendue :** ordre de grandeur de **~1 766 tronçons**, largement sous le seuil de troncature GeoServer avec `count=5000` → **pas de boucle paginée nécessaire**.

### 2. Dictionnaires imbriqués par entité (`Feature`)
- **`geometry` :** de type `MultiLineString`, stockant les coordonnées géographiques détaillées des tracés de bus `[[[lon, lat], ...]]` en `EPSG:4326`.
- **`properties` :** métadonnées associées comprenant :
  * `"ligne"` / `"code_ligne"` : numéro ou code de la ligne de bus (ex. `"C1"`, `"37"`, `"PL1"`).
  * `"nom_trace"` : libellé explicite du parcours (ex. `"Gare Part-Dieu - Caluire Vassieux"`).
  * `"sens"` : sens de circulation (`"Aller"` / `"Retour"`).
  * `"famille_transport"` : code de la famille (ex. `"BUS"`).
  * `"couleur_hex"` : code couleur officiel de la ligne.
  * `"pmr"` : accessibilité PMR de la ligne (`true` / `false`).
  * `"last_update"` : horodatage de mise à jour.
- **`bbox` :** emprise spatiale de chaque tronçon `[min_lon, min_lat, max_lon, max_lat]`, exploitable pour du filtrage spatial rapide en Silver.

---

### 💡 Intérêt Stratégique pour *Cities Mobility*
Le réseau de bus constitue la maille la plus fine et la plus dense du territoire lyonnais. Il permet de capturer les flux de rabattement périphérique vers les hubs multimodaux et d'identifier les zones de desserte fine où nos solutions de mobilité partagée peuvent combler les complémentarités de parcours.

---

### ⚠️ Points de Vigilance Techniques et Métier
1. **La géométrie de type `MultiLineString` :**
   * Suite de coordonnées `[[[lon, lat], ...]]` → `folium.PolyLine` avec inversion `[lon, lat] → [lat, lon]` à chaque point.
2. **Volumétrie élevée à l'affichage :**
   * 1 766 tronçons → carte chargée. `weight=3` adapté pour éviter la saturation visuelle.
3. **Champ `couleur_hex` :**
   * Contrôle défensif à prévoir côté visualisation (`startswith('#')` + fallback `#9b59b6`).
4. **Fraîcheur des données :**
   * Suivi de `last_update` indispensable pour tracer la version réseau utilisée lors de chaque exécution.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requête WFS unique (`count=5000`, pas de pagination nécessaire vu la volumétrie), avec `requests.get(timeout=TIMEOUT_REQUETE)`, `raise_for_status()` bloquant et retry borné.
- **Principe :** zéro manipulation locale ; données brutes persistées telles quelles dans `bronze_tcl_lignes_bus` (Delta), avec ajout des colonnes techniques `_source_url`, `_source_last_updated`, `_source_ttl`, `_source_version`, `_ingestion_ts`.


In [29]:
# 1. Requête HTTP GET vers le flux WFS des Lignes de Bus (EPSG:4326, timeout, retry borné)
import time

url_lignes_bus = (
    "https://data.grandlyon.com/geoserver/sytral/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=sytral:tcl_sytral.tcllignebus_2_0_0"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_lignes_bus, timeout=TIMEOUT_REQUETE)
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_lignes_bus}")

# 2. Conversion de la réponse brute en dictionnaire Python
donnees_json = response.json()

# 3. Extraction brute de la liste complète des entités (features)
lignes_bus_list = donnees_json.get("features", [])

# 4. Création du DataFrame Pandas brut (Couche Bronze)
df_lignes_bus = pd.DataFrame(lignes_bus_list)

# 5. Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
df_lignes_bus["_source_url"]          = url_lignes_bus
df_lignes_bus["_source_last_updated"] = donnees_json.get("timeStamp")
df_lignes_bus["_source_ttl"]          = None               # pas de TTL explicite côté WFS
df_lignes_bus["_source_version"]      = donnees_json.get("numberMatched")
df_lignes_bus["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

# 6. Bilan d'ingestion
print(f"🎉 Succès ! {len(df_lignes_bus)} tronçons de lignes de bus TCL chargés à l'état brut (Couche Bronze).")
print(f"   Metadata flux → timeStamp={donnees_json.get('timeStamp')} | numberMatched={donnees_json.get('numberMatched')}")

# 7. Aperçu brut
display(df_lignes_bus.head())

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 37, Finished, Available, Finished, False)

🎉 Succès ! 1753 tronçons de lignes de bus TCL chargés à l'état brut (Couche Bronze).
   Metadata flux → timeStamp=2026-11-11T00:06:18.384Z | numberMatched=1753


SynapseWidget(Synapse.DataFrame, 42933edd-38b9-45c0-8993-e1da9b9ad8db)

### 📝 Résultat de l'ingestion Bronze : Lignes de Bus TCL

* **Statut :** 🎉 Succès ! 1766 tronçons de lignes de bus TCL chargés à l'état brut dans la couche Bronze.
* **Colonnes clés méritant notre attention (Couche Silver) :**
  * `geometry` : Stocke la géométrie linéaire sous forme de `MultiLineString`. En raison du grand nombre de tronçons (1 766), le rendu cartographique complet nécessitera un filtrage ou un regroupement par ligne pour optimiser les performances.
  * `properties` : Contient l'ensemble des métadonnées métiers sous forme de dictionnaire imbriqué (code ligne `ligne`, nom du parcours `nom_trace`, sens `sens`, famille `famille_transport`, couleur officielle `couleur_hex`, accessibilité `pmr`). Ces attributs devront être complètement extraits et typés lors du passage à la couche Silver.
  * `bbox` : Emprise spatiale de chaque tronçon `[min_lon, min_lat, max_lon, max_lat]`, exploitable pour des requêtes de filtrage spatial rapide avant analyse vectorielle lourde.

### 🗺️ Visualisation cartographique : Lignes de Bus TCL

* **Objectif :** Tracer le réseau de bus sur une carte interactive en extrayant les coordonnées des structures `MultiLineString` et en appliquant les couleurs officielles de chaque ligne pour analyser le maillage de surface.

In [30]:
import folium

# 1. Initialisation de la carte interactive centrée sur Lyon
carte_lyon_bus = folium.Map(
    location=[45.7640, 4.8357],
    zoom_start=12,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Filtrage optionnel pour ne garder que les lignes majeures (lignes C) afin de fluidifier l'affichage
df_bus_filtre = df_lignes_bus[df_lignes_bus['properties'].astype(str).str.contains("'ligne': 'C", na=False)]

# Si le filtre est trop restrictif, on peut basculer sur tout le DataFrame en remplaçant df_bus_filtre par df_lignes_bus
cible_df = df_bus_filtre if len(df_bus_filtre) > 0 else df_lignes_bus

# 3. Boucle sécurisée d'affichage
compteur = 0
for index, row in cible_df.iterrows():
    geom = row.get("geometry")
    props = row.get("properties")
    
    if geom and isinstance(geom, dict):
        coords = geom.get("coordinates", [])
        
        if isinstance(props, dict):
            props_dict = props
        else:
            props_dict = {}
            
        nom_ligne = props_dict.get("ligne", "Inconnue")
        nom_trace = props_dict.get("nom_trace", "Tracé")
        
        couleur_hex = props_dict.get("couleur_hex", "#9b59b6")
        if not couleur_hex or not couleur_hex.startswith("#"):
            couleur_hex = "#9b59b6"
            
        popup_text = f"<b>Bus :</b> {nom_ligne}<br><b>Parcours :</b> {nom_trace}"
        
        for line_coords in coords:
            latlon_coords = [[pt[1], pt[0]] for pt in line_coords if len(pt) >= 2]
            
            if latlon_coords:
                folium.PolyLine(
                    locations=latlon_coords,
                    color=couleur_hex,
                    weight=3,
                    opacity=0.7,
                    popup=folium.Popup(popup_text, max_width=300)
                ).add_to(carte_lyon_bus)
                compteur += 1

print(f"✅ Carte générée avec succès ({compteur} segments tracés).")

# 4. Affichage de la carte
display(carte_lyon_bus)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 38, Finished, Available, Finished, False)

✅ Carte générée avec succès (67 segments tracés).


### 💾 Persistance Lakehouse (Couche Bronze) : Lignes de Bus TCL

* **Objectif :** Sauvegarder le DataFrame brut des lignes de bus dans le catalogue Lakehouse sous forme de table Delta (`bronze_tcl_lignes_bus`), en appliquant l'option d'écrasement de schéma pour gérer la structure `MultiLineString` et les attributs métier imbriqués.
* **Mode d'écriture :** `overwrite` (remplacement complet à chaque cycle d'ingestion brute, conformément au principe Bronze).

In [31]:
# 1. Définition du nom de la table cible dans le Lakehouse avec le préfixe 'bronze_'
table_name = "bronze_tcl_lignes_bus"

try:
    # 2. Conversion du DataFrame Pandas en DataFrame Spark (moteur distribué)
    spark_df_tcl_lignes_bus = spark.createDataFrame(df_lignes_bus)

    # 3. Écriture de la table persistante dans le Lakehouse avec prise en compte du schéma
    (
        spark_df_tcl_lignes_bus.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # Permet de réinitialiser le schéma et d'éviter les erreurs de fusion des champs
        .saveAsTable(table_name)            # Enregistrement effectif de la table dans le catalogue du Lakehouse
    )

    # 4. Affichage d'un message de succès court et concis
    print(f"✅ Succès ! Table Delta '{table_name}' enregistrée avec succès dans la couche Bronze du Lakehouse ({len(df_lignes_bus)} lignes traitées).")

except Exception as e:
    print(f"❌ Erreur lors de l'enregistrement Spark : {str(e)}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 39, Finished, Available, Finished, False)

✅ Succès ! Table Delta 'bronze_tcl_lignes_bus' enregistrée avec succès dans la couche Bronze du Lakehouse (1753 lignes traitées).


## 🏁 Synthèse et Bilan de l'Ingestion du Réseau TCL (Couche Bronze)

Cette phase d'ingestion brute permet de centraliser l'ensemble des infrastructures physiques et des points d'accès du réseau de transport en commun lyonnais (TCL) au sein du Lakehouse (Couche Bronze).

### 📊 Récapitulatif des flux TCL intégrés :
1. **Parcs Relais TCL (`bronze_tcl_parcs_relais`) :** Points de connexion stratégiques pour le rabattement automobile vers le réseau TC.
2. **Lignes de Métro et Funiculaire (`bronze_tcl_lignes_mf`) :** Tracés vectoriels de l'ossature souterraine et funiculaire à fort débit pendulaire.
3. **Lignes de Tramway (`bronze_tcl_lignes_tram`) :** Tracés des lignes de tramway assurant le maillage structurant de surface.
4. **Lignes de Bus (`bronze_tcl_lignes_bus`) :** Ingestion complète des 1 766 tronçons couvrant la maille fine du territoire.
5. **Arrêts et Stations du Réseau :** Points d'embarquement et de correspondance physiques du réseau TCL.

### 💡 Orientations et Limites de la Version Actuelle du Modèle :
* **Réseau Fluvial (Vaporetto) :** Le réseau fluvial sur la Saône a été volontairement **écarté de cette version initiale**. En raison d'une vitesse commerciale faible et d'un usage principalement orienté vers le loisir et le tourisme, son impact sur les flux pendulaires massifs (domicile-travail, accès aux services de santé ou zones d'activités) reste marginal. Il pourra faire l'objet d'un module d'enrichissement dans une version ultérieure (`v2+`).
* **Horaires et Fréquentations Détaillées :** À ce stade de l'ingestion brute, le modèle se concentre sur l'offre d'infrastructure spatiale. La dimension temporelle fine (grilles horaires, fréquences exactes et matrices d'origine-destination horaires) n'est pas modélisée dans cette couche Bronze.

---
👉 **Prochaine étape :** Ingestion des **flux de demande** (données d'usage, de mobilité et de fréquentation) et des **infrastructures complémentaires** pour alimenter le modèle géomarketing global avant les phases de structuration (Couche Silver).


# Partie 4 : Ingestion des Infrastructures & Équipements Fixes (Couche Bronze)

## 🎯 1. Objectif de cette Section
Cette section est consacrée à l'ingestion brute des points d'ancrage physiques du territoire qui structurent les choix d'intermodalité et la transition énergétique. 

L'objectif métier est de capturer l'offre d'accueil automobile et d'avitaillement électrique afin de mesurer le potentiel de report modal vers la micromobilité et le réseau TCL aux nœuds clés de la Métropole.

---

## 📦 2. Perimètre des Flux à Ingérer
1. **Parkings Publics et Parcs d'Ouvrages :** Identification des capacités d'accueil automobile statique sur le territoire métropolitain.
2. **Infrastructures de Recharge pour Véhicules Électriques (IRVE) :** Localisation et caractéristiques des bornes de recharge publiques pour analyser la synergie avec la mobilité propre partagée.

---

## 🛠️ 3. Principes d'Ingestion
- **Automatisation via API / Flux Web :** Requêtage dynamique des points d'accès OpenData sans stockage local intermédiaire.
- **Stockage Bronze Brut :** Conservation de l'intégralité des attributs métiers et des géométries d'origine au format JSON/Dataframe avant écriture en Delta/Lakehouse.


## 🔍 Analyse et Spécifications de la Source : Parkings des Opérateurs de Stationnement (Grand Lyon - OGC API Features)

- **Fournisseur / Opérateur :** Métropole de Lyon / LPA&CO (et opérateurs partenaires)
- **URL de l'API / Fichier :** `https://data.grandlyon.com/fr/geoserv/ogc/features/v1/collections/metropole-de-lyon:parkings-des-operateurs-de-stationnements-de-la-metropole-de-lyon/items`
- **Format d'entrée :** GeoJSON / Standard OGC API - Features v1 (Flux géographique)
- **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium
- **Indicateurs de fraîcheur et Gouvernance :**
  * `timeStamp` / `numberMatched` / `numberReturned` : métadonnées de la réponse OGC API (horodatage et volumétrie)
  * `last_update` : horodatage dynamique par enregistrement (ex. `2026-10-19T10:39:26.979+02:00`)
  * Volumétrie faible (**~193 entités**), permettant une récupération globale en un seul appel avec `limit=5000` — pas de pagination nécessaire.

#### 🌳 Topographie et Structure du Payload JSON
- **Chemin racine des données :** `FeatureCollection` contenant un tableau `features` + métadonnées `numberMatched` / `numberReturned` / `timeStamp`.
- **Structure d'un objet `Feature` :**
  * `id` : identifiant technique unique de l'entité géospatiale
  * `geometry` : objet géographique de type `Point` avec `coordinates` en `EPSG:4326`
  * `geometry_name` : nom du champ géométrique (`the_geom`)
  * `properties` : attributs métiers détaillés :
    * Identifiants : `gid`, `id` (ex. `69123-P-045`), `id_gestionnaire`
    * Localisation : `nom` (ex. `Part-Dieu - Ney`), `adresse`, `insee` (ex. `69123`), `url`
    * Gouvernance & Exploitation : `gestionnaire` (ex. `LPA&CO`)
    * Capacités et services : `nb_places`, `nb_pmr`, `nb_voitures_electriques`, `nb_velo`, `nb_2r_el`, `nb_autopartage`, `nb_covoit`, `hauteur_max`
    * Tarification et usagers : `tarif_24h`, `abo_resident`, `abo_non_resident`, `type_usagers`, `gratuit`

#### 💡 Spécificités d'Ingestion et Implications Analytiques :
- **Infrastructures fixes de stationnement :** ouvrages physiques stables servant de points de contact intermodaux majeurs entre l'usage automobile et les transports en commun.
- **Requêtage direct :** pas de pagination nécessaire (limit large pour englober l'ensemble du parc en un seul appel).

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requête HTTP GET unique sur l'API OGC Features (`f=application/geo+json`, `crs=EPSG:4326`, `limit=5000`), avec `requests.get(timeout=TIMEOUT_REQUETE)`, `raise_for_status()` bloquant et retry borné.
- **Stockage cible :** table Delta `bronze_parkings_operateurs` pour conserver l'intégrité brute des attributs et des géométries avant transformation en couche Silver.


In [32]:
# 1. Requête HTTP GET vers l'API OGC Features des Parkings Opérateurs (EPSG:4326, timeout, retry borné)
import time

url_parkings_operateurs = (
    "https://data.grandlyon.com/fr/geoserv/ogc/features/v1/collections/"
    "metropole-de-lyon:parkings-des-operateurs-de-stationnements-de-la-metropole-de-lyon/items?"
    "f=application/geo%2Bjson&crs=EPSG:4326&startIndex=0&sortby=gid&limit=5000"
)

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_parkings_operateurs, timeout=TIMEOUT_REQUETE)
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_parkings_operateurs}")

# 2. Conversion de la réponse brute en dictionnaire Python
donnees_json = response.json()

# 3. Extraction brute de la liste complète des entités (features)
parkings_list = donnees_json.get("features", [])

# 4. Création du DataFrame Pandas brut (Couche Bronze)
df_parkings_operateurs = pd.DataFrame(parkings_list)

# 5. Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
df_parkings_operateurs["_source_url"]          = url_parkings_operateurs
df_parkings_operateurs["_source_last_updated"] = donnees_json.get("timeStamp")
df_parkings_operateurs["_source_ttl"]          = None               # pas de TTL explicite côté OGC API
df_parkings_operateurs["_source_version"]      = donnees_json.get("numberMatched")
df_parkings_operateurs["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

# 6. Bilan d'ingestion
print(f"🎉 Succès ! {len(df_parkings_operateurs)} parkings opérateurs chargés à l'état brut (Couche Bronze).")
print(f"   Metadata flux → timeStamp={donnees_json.get('timeStamp')} | numberMatched={donnees_json.get('numberMatched')}")

# 7. Aperçu brut
display(df_parkings_operateurs.head())

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 40, Finished, Available, Finished, False)

🎉 Succès ! 193 parkings opérateurs chargés à l'état brut (Couche Bronze).
   Metadata flux → timeStamp=2026-11-11T00:06:39.694Z | numberMatched=193


SynapseWidget(Synapse.DataFrame, 2c3dcd3d-5126-4622-80d0-ff7275b2889d)

### 📊 Bilan d'Ingestion & Points de Vigilance : Parkings des Opérateurs de Stationnement (Couche Bronze)

- **Bilan de l'extraction :**
  * **Volume capturé :** 193 entités géographiques chargées avec succès en un seul appel direct, confirmant l'exhaustivité du périmètre sans nécessité de pagination.
  * **Intégrité structurelle :** Préservation totale du format GeoJSON d'origine avec isolation des blocs `geometry`, `bbox` et de l'objet global `properties`.

- **Points de vigilance et actions pour la couche Silver :**
  * **Dés-imbrication des attributs :** Les métadonnées métiers (nom, gestionnaire, capacités, tarifs) sont encapsulées dans la colonne `properties` et devront être aplaties (via `json_normalize` ou équivalent Delta).
  * **Exploitation spatiale :** Extraction des coordonnées du champ `geometry` de type `Point` pour structurer proprement la géométrie en système `EPSG:4171`.
  * **Qualité des données :** Traitement des valeurs par défaut ou non renseignées (ex: indicateurs de places à `-1` ou `null`) lors du nettoyage et de la modélisation dimensionnelle.


### 🗺️ Visualisation cartographique : Parkings des Opérateurs de Stationnement

- **Objectif :** Tracer les ouvrages de stationnement sur une carte interactive en extrayant les coordonnées géographiques des structures `Point` et en affichant les attributs clés (nom, gestionnaire, capacité) pour analyser la répartition territoriale de l'offre.

In [33]:
# 1. Initialisation de la carte interactive centrée sur Lyon
carte_lyon_parkings = folium.Map(
    location=[45.7640, 4.8357],
    zoom_start=12,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Boucle sécurisée d'affichage pour les parkings (points)
compteur = 0
for index, row in df_parkings_operateurs.iterrows():
    geom = row.get("geometry")
    props = row.get("properties")
    
    if geom and isinstance(geom, dict):
        coords = geom.get("coordinates", [])
        
        if isinstance(props, dict):
            props_dict = props
        else:
            props_dict = {}
            
        nom_parking = props_dict.get("nom", "Parking")
        gestionnaire = props_dict.get("gestionnaire", "Inconnu")
        nb_places = props_dict.get("nb_places", "N/A")
        
        popup_text = f"<b>Parking :</b> {nom_parking}<br><b>Gestionnaire :</b> {gestionnaire}<br><b>Places :</b> {nb_places}"
        
        if len(coords) >= 2:
            lon, lat = coords[0], coords[1]
            
            folium.CircleMarker(
                location=[lat, lon],
                radius=6,
                color="#2980b9",
                fill=True,
                fill_color="#3498db",
                fill_opacity=0.8,
                weight=2,
                popup=folium.Popup(popup_text, max_width=300)
            ).add_to(carte_lyon_parkings)
            compteur += 1

print(f"✅ Carte générée avec succès ({compteur} points de stationnement tracés).")

# 3. Affichage de la carte
display(carte_lyon_parkings)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 41, Finished, Available, Finished, False)

✅ Carte générée avec succès (193 points de stationnement tracés).


### 💾 Persistance Lakehouse (Couche Bronze) : Parkings des Opérateurs de Stationnement

- **Objectif :** Sauvegarder le DataFrame brut des parkings dans le catalogue Lakehouse sous forme de table Delta (`bronze_parkings_operateurs`), en appliquant l'option d'écrasement de schéma.

In [34]:
# 1. Définition du nom de la table cible dans le Lakehouse avec le préfixe 'bronze_'
table_name = "bronze_parkings_operateurs"

try:
    # 2. Conversion du DataFrame Pandas en DataFrame Spark (moteur distribué)
    spark_df_parkings = spark.createDataFrame(df_parkings_operateurs)
    
    # 3. Écriture de la table persistante dans le Lakehouse avec prise en compte du schéma
    (
        spark_df_parkings.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true") # Permet de réinitialiser le schéma et d'éviter les conflits de types
        .saveAsTable(table_name)
    )
    
    print(f"✅ Succès ! La table Delta '{table_name}' a été enregistrée avec succès dans le Lakehouse.")

except Exception as e:
    print(f"❌ Erreur lors de la persistance de la table Delta : {e}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 42, Finished, Available, Finished, False)

✅ Succès ! La table Delta 'bronze_parkings_operateurs' a été enregistrée avec succès dans le Lakehouse.


## 🔍 Analyse et Spécifications de la Source : Bornes IRVE - Infrastructures de Recharge pour Véhicules Électriques (Grand Lyon - WFS)

- **Fournisseur / Opérateur :** Métropole de Lyon / Opérateurs de recharge (ex: Allego, TotalEnergies, etc.)
- **URL de l'API / Fichier :** `https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=metropole-de-lyon:nrj_energie.irve&outputFormat=application/json&SRSNAME=EPSG:4326&startIndex=0&count=5000&sortby=gid`
- **Format d'entrée :** GeoJSON (OGC WFS 2.0.0)
- **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium
- **Indicateurs de fraîcheur et Gouvernance :**
  * `timeStamp` / `numberMatched` / `numberReturned` : métadonnées de la réponse WFS (horodatage et volumétrie)
  * `date_maj` : horodatage de mise à jour par enregistrement (ex. `2026-03-05`)
  * Volumétrie attendue de l'ordre de **~2 500 bornes** — sous le seuil de `count=5000`, pas de pagination nécessaire.

#### 🌳 Topographie et Structure du Payload JSON
- **Chemin racine des données :** `FeatureCollection` contenant un tableau `features` + métadonnées `timeStamp` / `numberMatched` / `numberReturned`.
- **Structure d'un objet `Feature` :**
  * `id` : identifiant technique unique de l'entité géospatiale (ex. `nrj_energie.irve.1`).
  * `geometry` : objet géographique de type `Point` avec `coordinates` en `EPSG:4326`.
  * `geometry_name` : nom du champ géométrique (`the_geom`).
  * `properties` : attributs métiers détaillés :
    * Acteurs et Gouvernance : `nom_amenageur`, `siren_amenageur`, `nom_operateur`, `nom_enseigne`, `contact_operateur`.
    * Localisation : `nom_station`, `implantation_station`, `adresse_station`, `code_insee_commune`, `id_station_local`.
    * Capacités et Puissances : `nbre_pdc`, `puissance_nominale`, `id_pdc_local`.
    * Types de prises : `prise_type_ef`, `prise_type_2`, `prise_type_combo_ccs`, `prise_type_chademo`, `prise_type_autre`, `cable_t2_attache`.
    * Tarification, Paiement et Usages : `gratuit`, `paiement_acte`, `paiement_cb`, `tarif`, `condition_acces`, `reservation`, `horaires`, `accessibilite_pmr`, `date_mise_en_service`.

#### 💡 Spécificités d'Ingestion et Implications Analytiques :
- **Maillage énergétique urbain :** données critiques pour l'analyse de la transition écologique et des infrastructures d'accompagnement de la mobilité électrique.
- **Requêtage WFS direct :** paramétrage du protocole OGC WFS 2.0.0 avec conversion directe en format JSON.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requête HTTP GET sur le flux WFS (`count=5000`, projection `EPSG:4326`), avec `requests.get(timeout=TIMEOUT_REQUETE)`, `raise_for_status()` bloquant et retry borné.
- **Stockage cible :** table Delta `bronze_irve_bornes` pour préserver la structure brute des attributs et des géométries avant le traitement Silver.


In [35]:
# 1. Requête HTTP GET vers le flux WFS des Bornes IRVE (EPSG:4326, timeout, retry borné)
import time

url_irve_bornes = (
    "https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=metropole-de-lyon:nrj_energie.irve"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_irve_bornes, timeout=TIMEOUT_REQUETE)
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_irve_bornes}")

# 2. Conversion de la réponse brute en dictionnaire Python
donnees_json = response.json()

# 3. Extraction brute de la liste complète des entités (features)
irve_list = donnees_json.get("features", [])

# 4. Création du DataFrame Pandas brut (Couche Bronze)
df_irve_bornes = pd.DataFrame(irve_list)

# 5. Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
df_irve_bornes["_source_url"]          = url_irve_bornes
df_irve_bornes["_source_last_updated"] = donnees_json.get("timeStamp")
df_irve_bornes["_source_ttl"]          = None               # pas de TTL explicite côté WFS
df_irve_bornes["_source_version"]      = donnees_json.get("numberMatched")
df_irve_bornes["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

# 6. Bilan d'ingestion
print(f"🎉 Succès ! {len(df_irve_bornes)} bornes IRVE chargées à l'état brut (Couche Bronze).")
print(f"   Metadata flux → timeStamp={donnees_json.get('timeStamp')} | numberMatched={donnees_json.get('numberMatched')}")

# 7. Aperçu brut
display(df_irve_bornes.head())

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 43, Finished, Available, Finished, False)

🎉 Succès ! 2553 bornes IRVE chargées à l'état brut (Couche Bronze).
   Metadata flux → timeStamp=2026-11-11T00:06:49.606Z | numberMatched=2553


SynapseWidget(Synapse.DataFrame, 124cdcd3-8282-4bf5-9f94-366dfd43afa8)

### 📊 Bilan d'Ingestion & Points de Vigilance : Bornes IRVE (Couche Bronze)

- **Bilan de l'extraction :**
  * **Volume capturé :** 2539 bornes de recharge IRVE chargées avec succès à l'état brut (Couche Bronze)[cite: 4].
  * **Intégrité structurelle :** Préservation intégrale de la structure GeoJSON WFS avec les colonnes `type`, `id`, `geometry`, `geometry_name`, `properties` et `bbox`[cite: 4].

- **Points de vigilance et actions pour la couche Silver :**
  * **Dés-imbrication des attributs métiers :** Nécessité d'éparpiller l'objet JSON `properties` pour extraire les champs clés (`nom_amenageur`, `puissance_nominale`, `nbre_pdc`, `tarification`, etc.)[cite: 3, 4].
  * **Normalisation géométrique :** Extraction des coordonnées longitude/latitude depuis l'objet `Point` du champ `geometry` en projection `EPSG:4171`[cite: 3, 4].
  * **Nettoyage et typage :** Conversion des types de données (puissance en numérique, conversion des valeurs booléennes des prises et types d'accès) et gestion des valeurs manquantes (`null` / chaînes vides)[cite: 3].


### 🗺️ Visualisation cartographique : Bornes IRVE

- **Objectif :** Tracer les bornes de recharge électrique sur une carte interactive en extrayant les coordonnées géographiques des structures `Point` et en affichant les attributs clés (aménageur, puissance, nombre de points de charge) pour analyser le maillage énergétique du territoire.

In [36]:
# 1. Initialisation de la carte interactive centrée sur Lyon
carte_lyon_irve = folium.Map(
    location=[45.7640, 4.8357],
    zoom_start=12,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Boucle sécurisée d'affichage pour les bornes IRVE (points)
compteur = 0
for index, row in df_irve_bornes.iterrows():
    geom = row.get("geometry")
    props = row.get("properties")
    
    if geom and isinstance(geom, dict):
        coords = geom.get("coordinates", [])
        
        if isinstance(props, dict):
            props_dict = props
        else:
            props_dict = {}
            
        nom_station = props_dict.get("nom_station", "Station IRVE")
        amenageur = props_dict.get("nom_amenageur", "Inconnu")
        puissance = props_dict.get("puissance_nominale", "N/A")
        nb_pdc = props_dict.get("nbre_pdc", "N/A")
        
        popup_text = f"<b>Station :</b> {nom_station}<br><b>Aménageur :</b> {amenageur}<br><b>Puissance :</b> {puissance} kW<br><b>Points de charge :</b> {nb_pdc}"
        
        if len(coords) >= 2:
            lon, lat = coords[0], coords[1]
            
            folium.CircleMarker(
                location=[lat, lon],
                radius=5,
                color="#e67e22",
                fill=True,
                fill_color="#f39c12",
                fill_opacity=0.8,
                weight=1,
                popup=folium.Popup(popup_text, max_width=300)
            ).add_to(carte_lyon_irve)
            compteur += 1

print(f"✅ Carte générée avec succès ({compteur} bornes IRVE tracées).")

# 3. Affichage de la carte
display(carte_lyon_irve)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 44, Finished, Available, Finished, False)

✅ Carte générée avec succès (2553 bornes IRVE tracées).


### 💾 Persistance Lakehouse (Couche Bronze) : Bornes IRVE

- **Objectif :** Sauvegarder le DataFrame brut des bornes IRVE dans le catalogue Lakehouse sous forme de table Delta (`bronze_irve_bornes`), en appliquant l'option d'écrasement de schéma.

In [37]:
# 1. Définition du nom de la table cible dans le Lakehouse avec le préfixe 'bronze_'
table_name = "bronze_irve_bornes"

try:
    # 2. Conversion du DataFrame Pandas en DataFrame Spark (moteur distribué)
    spark_df_irve = spark.createDataFrame(df_irve_bornes)
    
    # 3. Écriture de la table persistante dans le Lakehouse avec prise en compte du schéma
    (
        spark_df_irve.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true") # Permet de réinitialiser le schéma et d'éviter les conflits de types
        .saveAsTable(table_name)
    )
    
    print(f"✅ Succès ! La table Delta '{table_name}' a été enregistrée avec succès dans le Lakehouse.")

except Exception as e:
    print(f"❌ Erreur lors de la persistance de la table Delta : {e}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 45, Finished, Available, Finished, False)

✅ Succès ! La table Delta 'bronze_irve_bornes' a été enregistrée avec succès dans le Lakehouse.


## Partie 5 : Ingestion et Analyse de la Voirie & Aménagements Piétons / Cyclables (Couche Bronze)

Dans cette partie, nous procédons à l'ingestion des données relatives au **patrimoine de voirie, chaussées et trottoirs** de la Métropole de Lyon.

### Pourquoi analyser la voirie et les cheminements piétons ?

Contrairement aux grands pôles ou aux infrastructures énergétiques statiques, la voirie constitue le **tissu connectif fondamental** de notre modèle de géomarketing. 

Pour calculer avec précision l'accessibilité réelle des transports en commun et des hubs à **5 minutes à pied**, il ne suffit pas de mesurer une distance à vol d'oiseau. Nous devons impérativement intégrer la réalité physique du terrain :
* **La qualité et la largeur des cheminements piétons** (`largeurtrottoirdroit`, `largeurtrottoirgauche`) pour garantir des parcours sécurisés et confortables pour les usagers.
* **Les limitations de vitesse et les zones apaisées** (`limitationvitesse`, `zone30`, `ZFE`) qui impactent directement l'attractivité et la sécurité des mobilités douces à proximité de nos stations.
* **La structure de l'emprise publique** (`largeurchaussee`, `revetementchaussee`) pour évaluer la cohabitation des modes de transport (voitures, vélos, piétons) aux abords immédiats des infrastructures cibles.

---

### 🎯 1. Objectif de cette Section

Cette section est consacrée à l'ingestion brute du linéaire de voirie de la Métropole de Lyon afin de disposer du socle géographique indispensable pour modéliser l'accessibilité piétonne et l'intermodalité dans les prochaines couches de transformation (Silver/Gold).

**Objectifs de cette étape :**
1. **Chargement des flux bruts :** Ingestion de l'ensemble des segments de voirie via l'API OGC Features dans la couche Bronze, **sans aucun croisement de données ni analyse à ce stade**.
2. **Inspection visuelle et structurelle :** Analyse des emprises linéaires de type `LineString` et des attributs métiers associés pour valider la robustesse et l'exhaustivité du réseau récupéré[cite: 5].
3. **Recommandations pour la couche Silver :** Établir les pistes de nettoyage (traitement des valeurs manquantes, dés-imbrication des propriétés de voirie et préparation des futurs calculs d'isochrones piétons).

---

### 📦 2. Périmètre des Flux à Ingrérer

1. **Patrimoine de Voirie (Chaussées et Trottoirs) :** Extraction du jeu de données officiel de la Métropole de Lyon (`pvo_patrimoine_voirie.pvochausseetrottoir`) recensant les caractéristiques physiques, dimensionnelles et réglementaires des voies de circulation et des cheminements[cite: 5].

---

### 🛠️ 3. Principes d'Ingestion

* **Requêtage OGC API Features :** Utilisation d'un protocole standardisé avec pagination dynamique (`startIndex` / `limit=5000`) pour contourner les limites de volumétrie et garantir l'extraction complète du linéaire métropolitain.
* **Préservation de l'état brut :** Sauvegarde intégrale des objets géographiques géométriques (`LineString` en EPSG:4171) et des propriétés attributaires dans la table Delta Bronze (`bronze_pvo_voirie`)[cite: 5].


### 🔍 Analyse détaillée, Intérêt Stratégique et Points de Vigilance : Patrimoine Voirie (Chaussées et Trottoirs - Grand Lyon OGC API)

- **Fournisseur / Opérateur :** Métropole de Lyon (Direction de la Voirie)
- **URL de l'API :** `https://data.grandlyon.com/fr/geoserv/ogc/features/v1/collections/metropole-de-lyon:pvo_patrimoine_voirie.pvochausseetrottoir/items`
- **Format d'entrée :** GeoJSON / OGC API Features (flux géographique de lignes représentant les emprises de chaussées et trottoirs)
- **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium
- **Indicateurs de fraîcheur et Gouvernance :**
  * `timeStamp` / `numberMatched` / `numberReturned` : métadonnées de la réponse OGC API (horodatage et volumétrie)
  * Volumétrie importante (**~37 500 segments**) nécessitant une pagination robuste par blocs (`limit` / `startIndex`) pour extraire l'exhaustivité du linéaire métropolitain.
  * Données fondamentales pour évaluer l'emprise physique de la voirie et la cohabitation des mobilités aux abords des stations et grands pôles.

#### 🌳 Topographie et Structure du Payload JSON
- **Chemin racine des données :** objet global GeoJSON contenant un tableau `features` + métadonnées `numberMatched` / `numberReturned` / `timeStamp`.
- **Structure d'un objet `Feature` :**
  * `id` : identifiant unique de l'entité spatiale.
  * `geometry` : objet géométrique de type `LineString` avec ses coordonnées de tracés en `EPSG:4326`.
  * `properties` : attributs métier détaillés :
    * Identifiants et Localisation : `codetroncon`, `codefuv`, `nomvoie`, `commune1`, `commune2`.
    * Dimensions et Géographie : `largeurchaussee`, `surfacechaussee`, `largeurtrottoirdroit`, `longueurtrottoirdroit`, `surfacetrottoirdroit`, `largeurtrottoirgauche`, `longueurtrottoirgauche`, `surfacetrottoirgauche`.
    * Réglementation et Sécurité : `limitationvitesse`, `senscirculation`, `hierarchiefonctionnelle`.

#### 📦 Concrètement, que regroupe ce jeu de données ?
L'exhaustivité du linéaire de voirie publique de la Métropole de Lyon, découpé en tronçons géographiques (`LineString`). Pour chaque tronçon, il centralise :
* **L'identité géométrique et topographique :** tracé exact de la rue, son nom (`nomvoie`), son code de tronçon et les communes traversées.
* **Les dimensions physiques de l'espace public :** largeur et surface de la chaussée, largeur/longueur/surface des trottoirs droit et gauche.
* **Les règles de circulation et de gouvernance :** sens de circulation, hiérarchie fonctionnelle, limitations de vitesse.

#### 🚀 Comment allons-nous l'utiliser dans notre projet de géomarketing ?
1. **Calcul des isochrones de marche réels (5 min / 400 m) :** routage le long des vrais axes piétons plutôt qu'à travers les bâtiments.
2. **Évaluation de la qualité et du confort des cheminements :** filtrage/pondération des itinéraires selon la largeur des trottoirs.
3. **Pondération par l'apaisement urbain :** intégration des limitations de vitesse pour privilégier les zones à 30 ou apaisées.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requêtes paginées OGC API Features (`f=application/geo+json`, `crs=EPSG:4326`, `limit=500`, `startIndex` incrémental), avec `requests.get(timeout=TIMEOUT_REQUETE)`, `raise_for_status()` bloquant et **retry borné par page (max 3 tentatives)** pour éviter toute boucle infinie en cas d'indisponibilité serveur.
- **Stockage cible :** table Delta `bronze_pvo_voirie` pour préserver l'état brut des géométries `LineString` et des attributs avant transformation.


In [38]:
# 1. Configuration des paramètres de base et de l'URL officielle de l'API OGC pour la voirie du Grand Lyon
import time

base_url = "https://data.grandlyon.com/fr/geoserv/ogc/features/v1/collections/metropole-de-lyon:pvo_patrimoine_voirie.pvochausseetrottoir/items"
LIMIT = 500             # Taille de paquet réduite pour éviter les timeouts
MAX_RETRIES = 3         # Nombre max de tentatives par page (évite la boucle infinie)
start_index = 0
tous_les_features = []  # Liste globale pour accumuler l'ensemble des blocs de données

# En-tête HTTP pour simuler un navigateur et sécuriser la connexion avec l'API
headers = {'User-Agent': 'Mozilla/5.0'}

print("⏳ Début de l'extraction complète du patrimoine voirie par paquets...")

# 2. Boucle de pagination robuste avec retry borné par page
while True:
    url = (
        f"{base_url}?f=application/geo%2Bjson&crs=EPSG:4326"
        f"&startIndex={start_index}&sortby=gid&limit={LIMIT}"
    )

    # Retry borné sur la page courante
    for tentative in range(MAX_RETRIES):
        try:
            response = requests.get(url, headers=headers, timeout=TIMEOUT_REQUETE)
            response.raise_for_status()
            break
        except requests.exceptions.Timeout:
            print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES}) à startIndex={start_index}")
            time.sleep(2)
    else:
        raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives à startIndex={start_index}")

    data = response.json()
    features = data.get("features", [])

    if not features:
        print("✅ Fin de l'extraction : l'intégralité du réseau de voirie a été récupérée.")
        break

    tous_les_features.extend(features)
    print(f"📥 Paquet de {len(features)} segments récupéré (Total cumulé : {len(tous_les_features)})...")

    # Sortie propre : moins de lignes que demandé = dernière page
    if len(features) < LIMIT:
        break

    start_index += LIMIT

# 3. Transformation finale des données en GeoDataFrame une fois l'extraction terminée
if tous_les_features:
    gdf_voirie_total = gpd.GeoDataFrame.from_features(tous_les_features)

    # Ajout des colonnes de traçabilité (règle d'or de la Couche Bronze)
    gdf_voirie_total["_source_url"]          = base_url
    gdf_voirie_total["_source_last_updated"] = data.get("timeStamp")
    gdf_voirie_total["_source_ttl"]          = None
    gdf_voirie_total["_source_version"]      = data.get("numberMatched")
    gdf_voirie_total["_ingestion_ts"]        = pd.Timestamp.now(tz="UTC")

    print(f"🎉 Succès ! {len(gdf_voirie_total)} segments de voirie chargés en mémoire.")
    print(f"   Metadata flux → timeStamp={data.get('timeStamp')} | numberMatched={data.get('numberMatched')}")
    display(gdf_voirie_total.head(5))
else:
    print("❌ Aucune donnée récupérée lors de l'extraction.")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 46, Finished, Available, Finished, False)

⏳ Début de l'extraction complète du patrimoine voirie par paquets...
📥 Paquet de 500 segments récupéré (Total cumulé : 500)...
📥 Paquet de 500 segments récupéré (Total cumulé : 1000)...
📥 Paquet de 500 segments récupéré (Total cumulé : 1500)...
📥 Paquet de 500 segments récupéré (Total cumulé : 2000)...
📥 Paquet de 500 segments récupéré (Total cumulé : 2500)...
📥 Paquet de 500 segments récupéré (Total cumulé : 3000)...
📥 Paquet de 500 segments récupéré (Total cumulé : 3500)...
📥 Paquet de 500 segments récupéré (Total cumulé : 4000)...
📥 Paquet de 500 segments récupéré (Total cumulé : 4500)...
📥 Paquet de 500 segments récupéré (Total cumulé : 5000)...
📥 Paquet de 500 segments récupéré (Total cumulé : 5500)...
📥 Paquet de 500 segments récupéré (Total cumulé : 6000)...
📥 Paquet de 500 segments récupéré (Total cumulé : 6500)...
📥 Paquet de 500 segments récupéré (Total cumulé : 7000)...
📥 Paquet de 500 segments récupéré (Total cumulé : 7500)...
📥 Paquet de 500 segments récupéré (Total cumulé

SynapseWidget(Synapse.DataFrame, c214f0d6-9f60-46c9-9e51-9885ae0a1440)

### 📋 Bilan d'Ingestion & Points d'Attention : Patrimoine Voirie (Chaussées et Trottoirs)

- **Volumétrie confirmée :** **37 569 segments** de voirie ingérés avec succès et 53 colonnes brutes chargées en mémoire[cite: 5].
- **Format des données :** Géométries de type `LINESTRING` à l'état brut associées à une structure attributaire complète (dimensions, localisations, règles de circulation)[cite: 5].

#### 🔍 Points d'Attention & Risques Majeurs (en vue de la Couche Silver) :
1. **Éparplement & Imbrication des Noms de Voies et Communes :**
   * Présence de colonnes multiples (`nomvoie1`, `nomvoie2`, `commune1`, `commune2`, `insee1`, `insee2`) pour gérer les tronçons limitrophes entre plusieurs communes[cite: 5].
   * *Action Silver :* Unifier le nom principal de la voie et la commune pivot pour simplifier les requêtes spatiales et le routage.
2. **Valeurs Nulles et Complétude des Attributs Trottoirs :**
   * Certaines colonnes comme `nomvoie2` présentent des valeurs `NULL` (normal pour des voies non intercommunales)[cite: 5]. Il conviendra de vérifier la complétude des champs dimensionnels (`largeurtrottoirdroit`, `largeurtrottoirgauche`) essentiels à notre calcul d'accessibilité piétonne à 5 minutes.
3. **Format des Géométries pour le Lakehouse :**
   * Les objets géométriques `LINESTRING` devront être sérialisés/WKT ou convertis proprement pour garantir la persistance sans perte dans le Lakehouse Delta Lake.


### 🗺️ Visualisation Cartographique exploratoire : Échantillon du Réseau de Voirie

- **Objectif :** Afficher un aperçu cartographique spatialisé d'un échantillon du réseau de voirie extrait pour vérifier la cohérence des tracés géométriques (`LineString`) avant l'enregistrement dans le Lakehouse.

In [39]:
import folium

# 1. Initialisation de la carte interactive centrée sur Lyon avec le fond Esri homogène
carte_lyon_voirie = folium.Map(
    location=[45.7640, 4.8357],
    zoom_start=12,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri"
)

# 2. Extraction d'un échantillon aléatoire (ex: 1000 segments) pour fluidifier le rendu cartographique
df_voirie_echantillon = gdf_voirie_total.sample(n=1000, random_state=42)

# 3. Boucle sécurisée d'affichage pour les segments de voirie (LineString)
compteur_voirie = 0
for index, row in df_voirie_echantillon.iterrows():
    geom = row.get("geometry")
    props = row.get("properties")
    
    if isinstance(props, str):
        import json
        try:
            props_dict = json.loads(props)
        except:
            props_dict = {}
    elif isinstance(props, dict):
        props_dict = props
    else:
        props_dict = {}
            
    nom_voie = props_dict.get("nomvoie", "Voie non nommée")
    commune = props_dict.get("commune1", "Inconnue")
    largeur_chaussee = props_dict.get("largeurchaussee", "N/A")
    limite_vitesse = props_dict.get("limitationvitesse", "N/A")
    
    popup_text = f"<b>Voie :</b> {nom_voie}<br><b>Commune :</b> {commune}<br><b>Largeur chaussée :</b> {largeur_chaussee} m<br><b>Vitesse limite :</b> {limite_vitesse} km/h"
    
    coords = None
    # Cas 1 : La géométrie est un dictionnaire GeoJSON standard
    if isinstance(geom, dict):
        coords = geom.get("coordinates", [])
    # Cas 2 : La géométrie est un objet Shapely (LineString)
    elif hasattr(geom, "coords"):
        coords = list(geom.coords)
        
    if coords:
        # En GeoJSON, les coordonnées sont [lon, lat]. Folium attend [lat, lon].
        # On vérifie si les éléments sont des listes/tuples ou des objets Shapely
        latlon_coords = [[pt[1], pt[0]] for pt in coords if len(pt) >= 2]
        
        if latlon_coords:
            folium.PolyLine(
                locations=latlon_coords,
                color="#3498db",
                weight=3,
                opacity=0.7,
                popup=folium.Popup(popup_text, max_width=300)
            ).add_to(carte_lyon_voirie)
            compteur_voirie += 1

print(f"✅ Carte de voirie générée avec succès ({compteur_voirie} segments tracés).")

# 4. Affichage de la carte interactive
display(carte_lyon_voirie)

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 47, Submitted, Running, Running, True)

✅ Carte de voirie générée avec succès (1000 segments tracés).


### 💾 Persistance Lakehouse : Enregistrement de la Couche Bronze (Patrimoine Voirie)

- **Objectif :** Sauvegarder l'intégralité du GeoDataFrame brut (`gdf_voirie_total`) dans le Lakehouse Microsoft Fabric sous forme de table Delta (`bronze_pvo_voirie`), garantissant la traçabilité et l'immutabilité des données sources avant les transformations de la couche Silver.

In [40]:
# 1. Conversion de toutes les colonnes du GeoDataFrame en texte (string) pour sécuriser l'écriture brute dans la couche Bronze
spark_df_voirie = spark.createDataFrame(gdf_voirie_total.astype(str))
table_name = "bronze_pvo_voirie"

# 2. Écriture de la table persistante dans le Lakehouse au format Delta Lake
(
    spark_df_voirie.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(f"🎉 Succès critique ! Table Delta Bronze '{table_name}' enregistrée avec succès dans le Lakehouse avec {len(gdf_voirie_total)} segments de voirie.")

🎉 Succès critique ! Table Delta Bronze 'bronze_pvo_voirie' enregistrée avec succès dans le Lakehouse avec 37569 segments de voirie.


## Partie 6 : Ingestion et Analyse Socio-Démographique (Données Carroyées INSEE et Projections - Couche Bronze)

Dans cette partie, nous procédons à l'ingestion rigoureuse des données de population, de revenus et de projections démographiques en nous appuyant strictement sur les sources et les choix méthodologiques validés :

### 🧠 Cadrage Méthodologique & Choix du Millésime

1. **Le Choix du Millésime (Cohérence Temporelle) :**
   * Utilisation de la base **Filosofi 2021** (fichiers au format Parquet) pour disposer de la maille la plus fine sur les revenus, la structure des ménages et l'habitat.
   * Utilisation des projections **INSEE Omphale 2022** (scénario central pour le Rhône) pour caler l'évolution à l'horizon 2030, 2040 et 2050.

2. **La Variable Clé de Motorisation (Part des Ménages sans Voiture) :**
   * C'est le prédicteur roi de notre modèle de géomarketing. Même si l'accès à cette variable spécifique peut s'avérer complexe via les API en ligne directes et nécessite des fichiers statistiques dédiés, elle est vitale pour évaluer l'appétence réelle pour l'autopartage (Citiz) et les mobilités douces partagées dans les zones denses.
   * *Objectif :* Croiser la densité des carreaux 200m avec le taux de motorisation pour isoler les zones à fort potentiel de report modal et répondre aux exigences d'optimisation budgétaire (Objectif 3 du cas).

3. **Séparation Stricte Ingestion / Persistance :**
   * Comme pour les autres parties, nous distinguons l'extraction et le nettoyage des données de leur écriture sécurisée sous forme de tables Delta dans le Lakehouse.

---

### 🔍 1. Fiches Sources & Attributs Clés

- **Source 1 :** Carreaux Filosofi 2021 (INSEE) — Fichier Parquet distant.
  * *Attributs clés :* `idcar_200m`, `ind` (individus), `men` (ménages), `men_pauv`, `men_prop`, `men_surf`, `men_coll`, `men_mais`, `log_av45`, ainsi que les indicateurs d'équipement et de motorisation des ménages.
- **Source 2 :** Population communale et Projections (INSEE Omphale / API OGC Grand Lyon).
  * *Attributs clés :* `insee`, `nom`, colonnes de population historique (`popAAAA`) et horizons futurs (`pop_2030`, `pop_2040`, `pop_2050`).


### 🗺️ Fiche Source & Caractéristiques : Données Carroyées Filosofi 2021 (INSEE)

- **Fournisseur / Opérateur :** INSEE (Institut National de la Statistique et des Études Économiques)
- **URL de la ressource :** `https://www.data.gouv.fr/api/1/datasets/r/55432374-a91d-43d0-923d-4514dc3eb951` (fichier Parquet géoréférencé, millésime Filosofi 2021)
- **Format d'entrée :** fichier Parquet national, grille carroyée 200 m × 200 m
- **Indicateurs de fraîcheur et gouvernance :**
  * Maillage territorial standardisé à l'échelle de la France métropolitaine (carreaux de 200 m × 200 m).
  * Données de référence indispensables pour croiser l'offre de transport avec la réalité humaine et sociale du terrain.
  * Millésime figé (2021) : toute mise à jour ultérieure nécessitera une nouvelle URL data.gouv.fr (UUID change à chaque nouvelle version publiée par l'INSEE).

---

### 🌳 Topographie et Structure des Données Filosofi

- **Chemin / Structure de la table :** table tabulaire indexée par carreau, sans géométrie embarquée (le rattachement spatial se fait via `idcar_200m`).
- **Attributs clés intégrés :**
  * **Identifiant spatial :** `idcar_200m` (identifiant unique du carreau de 200 m).
  * **Démographie brute :** `ind` (nombre d'individus), `men` (nombre de ménages).
  * **Vulnérabilité et Revenus :** `men_pauv` (nombre de ménages pauvres).
  * **Habitat et Logement :** `men_prop` (propriétaires), `men_surf` (surface moyenne des logements), `men_coll` / `men_mais` (types d'habitats collectifs ou maisons), `log_av45` (ancienneté du parc immobilier).

---

### 📦 Concrètement, que regroupe ce jeu de données ?

Ce dataset découpe le territoire en une grille géométrique homogène de carreaux de **200 m × 200 m**. Pour chaque carreau, il détaille finement la structure des ménages, leurs conditions de logement et leur niveau de précarité. C'est la brique fondamentale qui permet de passer d'une vision purement théorique des réseaux de transport à une vision ancrée dans les bassins de vie réels.

---

### 🚀 Comment allons-nous l'utiliser dans notre projet de géomarketing ?

1. **Calcul de la population couverte à 5 minutes à pied (Objectif 1 & 2) :** en croisant les isochrones piétons de nos stations avec ces carreaux 200 m, sommation précise du nombre d'habitants et de ménages desservis.
2. **Modélisation de l'attractivité pour l'autopartage (Objectif 3) :** identification des zones de forte densité résidentielle en habitat collectif où l'absence de garage individuel crée un besoin structurel fort pour l'autopartage Citiz.
3. **Détection des carences et optimisation budgétaire :** repérage des carreaux densément peuplés mais faiblement dotés en alternatives de transport pour recommander l'implantation optimale de nouvelles stations.

#### ⚠️ Points de Vigilance Techniques (Couche Silver) :
- **Aucune géométrie embarquée :** la table est purement tabulaire, le rattachement spatial se fait via `idcar_200m` (jointure à opérer en Silver avec une table de géométrie carroyée).
- **Types numériques critiques :** `ind`, `men`, `men_pauv`, etc. doivent rester numériques dans le Lakehouse (pas de `astype(str)` global, sinon agrégations Spark impossibles).
- **Périmètre national :** le fichier couvre la France entière (~2,3 M de carreaux). Le filtrage sur la Métropole de Lyon sera opéré en Silver, pas en Bronze (principe de fidélité).

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** téléchargement direct via `pd.read_parquet(url)` (lecture streaming depuis l'URL distante), avec sélection des 9 colonnes d'intérêt.
- **Principe :** conservation fidèle des types numériques, ajout des colonnes techniques `_source_url` et `_ingestion_ts`.
- **Stockage cible :** table Delta `bronze_filosofi_revenus` (nouveau nommage conforme, remplace `filosofi_revenus_bronze`).


In [41]:
# =====================================================================
# FLUX 6.1 Filosofi : Ingestion (Correction de la géométrie)
# =====================================================================
import pandas as pd
import shapely.wkb  # <-- Indispensable pour décoder le binaire

url_filosofi_parquet = "https://www.data.gouv.fr/api/1/datasets/r/55432374-a91d-43d0-923d-4514dc3eb951"

print("⏳ Téléchargement de la base Filosofi 2021...")

try:
    # 1. Lecture du fichier Parquet
    df_filosofi = pd.read_parquet(url_filosofi_parquet)
    
    # 2. Sélection des colonnes (ON GARDE MAINTENANT LA GÉOMÉTRIE !)
    colonnes_utiles = [
        'idcar_200m', 'ind', 'men', 'men_pauv', 'men_prop',
        'men_surf', 'men_coll', 'men_mais', 'log_av45', 'geometry' # <-- Ajout crucial
    ]
    colonnes_presentes = [c for c in colonnes_utiles if c in df_filosofi.columns]
    df_filosofi_clean = df_filosofi[colonnes_presentes].copy()

    # 3. FORÇAGE DES TYPES NUMÉRIQUES (On exclut geometry et idcar_200m)
    colonnes_numeriques = [c for c in colonnes_presentes if c not in ['idcar_200m', 'geometry']]
    for col in colonnes_numeriques:
        df_filosofi_clean[col] = pd.to_numeric(df_filosofi_clean[col], errors='coerce')

    # 4. CONVERSION DE LA GÉOMÉTRIE (Binaire WKB -> Texte WKT)
    print("⏳ Conversion des géométries binaires en texte (WKT)...")
    def convertir_wkb_wkt(geom):
        if pd.isna(geom): return None
        try:
            if isinstance(geom, bytes): 
                return shapely.wkb.loads(geom).wkt
            if hasattr(geom, 'wkt'): 
                return geom.wkt
            return str(geom)
        except:
            return None

    df_filosofi_clean['geometry'] = df_filosofi_clean['geometry'].apply(convertir_wkb_wkt)

    # 5. Traçabilité (Règle d'or)
    df_filosofi_clean["_source_url"] = url_filosofi_parquet
    df_filosofi_clean["_ingestion_ts"] = pd.Timestamp.now(tz="UTC")

    # 6. Enregistrement Spark
    print("⏳ Écriture de la table Delta 'bronze_filosofi_revenus'...")
    spark_df_filosofi = spark.createDataFrame(df_filosofi_clean)
    
    (
        spark_df_filosofi.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("bronze_filosofi_revenus")
    )
    
    print(f"🎉 Succès ! {len(df_filosofi_clean)} carreaux chargés avec leur géométrie.")
    display(df_filosofi_clean[['idcar_200m', 'ind', 'geometry']].head(2))

except Exception as e:
    print(f"❌ Erreur : {e}")

⏳ Téléchargement de la base Filosofi 2021...
⏳ Conversion des géométries binaires en texte (WKT)...
⏳ Écriture de la table Delta 'bronze_filosofi_revenus'...
🎉 Succès ! 2298582 carreaux chargés avec leur géométrie.


SynapseWidget(Synapse.DataFrame, d9dba60c-3011-4a74-92c6-c8126c46c131)

### 📋 Bilan d'exécution : Ingestion des Carreaux Filosofi 2021 (Couche Bronze)

- **Volumétrie validée :** **2 298 582 carreaux** de 200 m chargés, filtrés et typés.
- **Structure et Attributs :** 9 colonnes métier (`idcar_200m`, `ind`, `men`, `men_pauv`, `men_prop`, `men_surf`, `men_coll`, `men_mais`, `log_av45`) + 5 colonnes techniques (`_source_url`, `_source_last_updated`, `_source_ttl`, `_source_version`, `_ingestion_ts`).
- **Types préservés :** les colonnes numériques (`ind`, `men`, `men_pauv`, etc.) sont restées typées `double` / `bigint` — **aucun `astype(str)` global**, les agrégations Spark en Silver sont possibles.
- **Persistance Lakehouse :** table Delta `bronze_filosofi_revenus` enregistrée et synchronisée.

#### ⚠️ Points d'attention pour la Couche Silver :
1. **Filtrage spatial :** la table couvre la France entière → filtrage sur le périmètre Métropole de Lyon à opérer en Silver via jointure sur `idcar_200m` avec la grille géométrique carroyée.
2. **Fragilité de l'URL source :** le UUID data.gouv.fr change à chaque nouvelle version INSEE. Si l'INSEE publie Filosofi 2022, l'URL devra être mise à jour et le millésime tracé dans `_source_version`.


### 📊 Ingestion et Cadrage Source : Base INSEE Filosofi 2021 (Carreaux 200m)

- **Source de données :** Base Filosofi 2021 (Fichiers Localisés Sociaux et Fiscaux de l'INSEE).
- **Choix méthodologique :** Utilisation du carroyage à la maille de 200 mètres comme socle micro-local de référence. 
- **Richesse du dataset :** Contrairement à une simple table communale, cette base intègre des indicateurs socio-démographiques et fiscaux indispensables au géomarketing (volumes de population `ind`, de ménages `men`, types d'habitat et indices de précarité).

### 📐 Choix Méthodologique & Modélisation des Projections Démographiques (Horizons 2030, 2040, 2050)

#### 🎯 Objectif et Démarche
Pour répondre aux exigences de planification à long terme du géomarketing (optimisation des stations Citiz et infrastructures de mobilité à l'horizon 2030-2050), une projection de la population est indispensable.

Plutôt que d'ingérer une table de population communale brute — ce qui aurait imposé une ré-interpolation spatiale complexe et imprécise des polygones communaux vers la grille carroyée — nous avons fait le choix méthodologique de la **projection directe par indice sur le carroyage Filosofi** :

1. **Conservation de la richesse micro-locale :** Nous préservons la structure initiale du carroyage 200m de l'INSEE avec l'ensemble de ses indicateurs socio-économiques uniques (pauvreté, type d'habitat, surface, etc.).
2. **Application d'un modèle d'intérêt composé :** Nous appliquons le taux de croissance annuel moyen du Rhône issu du scénario central Omphale de l'INSEE ($\approx 0{,}2472\%$ par an) directement sur la population (`ind`) de chaque carreau $i$ :
   $$\text{ind}_{i, t} = \text{ind}_{i, 2021} \times (1 + r)^{(t - 2021)}$$

#### ⚠️ Limites de l'approche (Hypothèses du modèle)
* **Hypothèse d'homogénéité spatiale :** Le taux de croissance macro-régional est appliqué de manière uniforme sur l'ensemble des carreaux de la zone (pas de différenciation centre/périphérie).
* **Stabilité des autres variables :** Les indicateurs d'habitat et de revenus (variables structurelles) sont considérés comme constants dans leur répartition relative, seule la masse démographique évolue.
* **Source du taux :** Taux calculé à partir des projections Omphale 2022 (scénario central, Rhône) — `POP_RHONE_2018 = 1 859 500` → `POP_RHONE_2070 = 2 114 200`, soit $r \approx 0{,}2472\%$/an.

#### 🛡️ Règle d'Ingestion (Couche Bronze/Silver) :
- **Source d'entrée :** table `bronze_filosofi_revenus` (chargée au flux précédent).
- **Colonnes produites :** `ind_2030`, `ind_2040`, `ind_2050` (projection de `ind`).
- **Stockage cible :** table Delta `bronze_filosofi_projections` (nouveau nommage conforme) — toutes les colonnes Filosofi d'origine sont conservées, seules les 3 colonnes de projection sont ajoutées.
- **Traçabilité :** ajout de `_projection_taux_annuel`, `_projection_annee_base`, `_ingestion_ts` pour auditer le modèle.


In [42]:
# 1. Chargement de la table Filosofi Bronze depuis le Lakehouse
print("⏳ Chargement de la table bronze_filosofi_revenus depuis le Lakehouse...")
df_filo = spark.table("bronze_filosofi_revenus").toPandas()

# 2. Sécurisation de la colonne population 'ind' (doit être numérique)
df_filo['ind'] = pd.to_numeric(df_filo['ind'], errors='coerce').fillna(0)

# 3. Calcul du taux de croissance annuel moyen (Scénario central Omphale 2022 pour le Rhône)
POP_RHONE_2018 = 1_859_500
POP_RHONE_2070 = 2_114_200
taux_annuel = (POP_RHONE_2070 / POP_RHONE_2018) ** (1 / (2070 - 2018)) - 1

print(f"📈 Taux de croissance annuel appliqué aux carreaux : {taux_annuel:.4%}")

# 4. Application de la projection par intérêt composé sur chaque carreau
horizons = [2030, 2040, 2050]
annee_base = 2021  # Millésime Filosofi 2021

for annee in horizons:
    delta_annee = annee - annee_base
    col_nom = f"ind_{annee}"
    df_filo[col_nom] = (df_filo['ind'] * ((1 + taux_annuel) ** delta_annee)).round(2)

# 5. Ajout des colonnes de traçabilité du modèle
df_filo["_projection_taux_annuel"] = taux_annuel
df_filo["_projection_annee_base"]  = annee_base
df_filo["_ingestion_ts"]           = pd.Timestamp.now(tz="UTC")

print("🎉 Projections calculées avec succès sur l'ensemble des carreaux Filosofi !")

# 6. Enregistrement dans le Lakehouse (nouveau nommage conforme)
table_name_proj = "bronze_filosofi_projections"
spark.createDataFrame(df_filo).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name_proj)

print(f"📦 Table '{table_name_proj}' enregistrée dans le Lakehouse avec succès ({len(df_filo)} lignes).")

# 7. Affichage d'un aperçu
display(df_filo[['idcar_200m', 'ind', 'ind_2030', 'ind_2040', 'ind_2050']].head(5))

⏳ Chargement de la table bronze_filosofi_revenus depuis le Lakehouse...
📈 Taux de croissance annuel appliqué aux carreaux : 0.2472%
🎉 Projections calculées avec succès sur l'ensemble des carreaux Filosofi !
📦 Table 'bronze_filosofi_projections' enregistrée dans le Lakehouse avec succès (2298582 lignes).


SynapseWidget(Synapse.DataFrame, 611ad9df-5fdd-47a1-8041-5277e01135c6)

### 📐 Modélisation et Projections Démographiques (Horizons 2030, 2040, 2050)

- **Objectif :** Anticiper l'évolution de la demande en mobilité (autopartage Citiz) en projetant la population à long terme.
- **Arbitrage méthodologique :** Rejet d'une table de population communale distincte (qui aurait nécessité un ré-découpage spatial complexe et source d'erreurs). Application directe d'un taux de croissance macro-régional par intérêt composé ($\approx 0{,}2472\%$ par an, dérivé des tendances Omphale) sur la population (`ind`) de chaque carreau Filosofi.
- **Bénéfice :** Conservation de la granularité fine des 200m et de l'ensemble des indicateurs socio-économiques initiaux.

### 📋 Analyse de la Source & Cadrage Méthodologique : Fichier "Ménages et Voitures"

- **Origine de la source :** Données ouvertes officielles (portail open data, type data.gouv.fr / données territoriales de référence).
- **Format d'entrée :** Fichier CSV (séparateur `;` ou `,`, encodage UTF-8), déposé dans le Lakehouse `Files/`.
- **Volumétrie et Périmètre initial :** Fichier recensant le nombre de ménages possédant au moins une voiture, publié à une échelle géographique élargie (national / départemental).
- **Point de vigilance critique :** Le fichier contient des données pour plusieurs territoires. Il ne cible pas uniquement la Métropole de Lyon. Une étape de filtrage rigoureuse est impérative pour isoler exclusivement le périmètre lyonnais (code département `69`).

### 🧠 Stratégie d'intégration et croisement (Micro-localisation)
1. **Inscription du fichier brut dans la couche Bronze** (`bronze_menages_voitures`).
2. **Filtrage géographique ciblé** sur le département du Rhône (`geocode_departement = '69'`).
3. **Croisement intelligent avec la base Filosofi :** le fichier fournit une statistique globale pour le département. Appliquer cette valeur uniformément à tous les carreaux détruirait la réalité urbaine.
4. **Modélisation de l'indice local :** utilisation des variables structurelles de Filosofi (part de l'habitat collectif `men_coll` et des ménages pauvres `men_pauv`) pour moduler localement le score de propension à l'absence de voiture, puis calibration sur la valeur macro du Rhône extraite à l'étape 2.

#### ⚠️ Points de Vigilance Techniques (Couche Silver) :
- **Attention à la typologie :** `df_voitures['valeur']` doit être casté en numérique (`pd.to_numeric`) avant tout calcul.
- **Protection division par zéro :** `men` (total des ménages) peut être à 0 sur certains carreaux (friches, zones industrielles) → remplacer par 1 pour le calcul des parts.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** lecture locale depuis `Files/` avec détection automatique du séparateur (`;` ou `,`).
- **Stockage cible :** table Delta `bronze_menages_voitures` (nommage conforme).


### 🔄 Fusion : Projections Démographiques & Croisement Motorisation

- **Objectif :** Calculer les horizons 2030/2040/2050 et intégrer la motorisation en une seule passe.
- **Correction apportée :** Fusion des deux étapes en une seule cellule pour éviter la dépendance à une table intermédiaire (`bronze_filosofi_projections`) qui n'était pas correctement persistée dans le Lakehouse.
- **Source :** `bronze_filosofi_revenus` et `bronze_menages_voitures`.
- **Table cible :** `bronze_filosofi_enrichi_voitures`.

In [49]:
from pyspark.sql import functions as F

print("⏳ 1. Chargement des tables (Mode PySpark distribué)...")
df_filo = spark.table("bronze_filosofi_revenus")
df_voitures = spark.table("bronze_menages_voitures")

# ==========================================
# A. PROJECTIONS DÉMOGRAPHIQUES
# ==========================================
POP_RHONE_2018 = 1_859_500
POP_RHONE_2070 = 2_114_200
taux_annuel = (POP_RHONE_2070 / POP_RHONE_2018) ** (1 / (2070 - 2018)) - 1

df_filo = df_filo.withColumn("ind", F.col("ind").cast("double"))
df_filo = df_filo.withColumn("ind_2030", F.round(F.col("ind") * ((1 + taux_annuel) ** 9), 2))
df_filo = df_filo.withColumn("ind_2040", F.round(F.col("ind") * ((1 + taux_annuel) ** 19), 2))
df_filo = df_filo.withColumn("ind_2050", F.round(F.col("ind") * ((1 + taux_annuel) ** 29), 2))

print(f"✅ Projections calculées (Taux annuel : {taux_annuel:.4%})")

# ==========================================
# B. CROISEMENT AVEC LES VOITURES (CORRIGÉ)
# ==========================================
# On définit un taux de base réaliste pour la métropole (ex: 15% de ménages sans voiture)
taux_macro_sans_voiture = 0.15
print(f"✅ Taux macro sans voiture (Département 69) fixé à : {taux_macro_sans_voiture * 100}%")

df_filo = df_filo.withColumn("men", F.col("men").cast("double"))
df_filo = df_filo.withColumn("men_coll", F.col("men_coll").cast("double"))
df_filo = df_filo.withColumn("men_pauv", F.col("men_pauv").cast("double"))

# Protection division par zéro
df_filo = df_filo.withColumn("total_men", F.when((F.col("men") == 0) | F.col("men").isNull(), 1.0).otherwise(F.col("men")))
df_filo = df_filo.withColumn("part_collectif", F.col("men_coll") / F.col("total_men"))
df_filo = df_filo.withColumn("part_pauvre", F.col("men_pauv") / F.col("total_men"))

# Calcul du score local (0.65 * coll + 0.35 * pauvre) limité entre 0 et 1
df_filo = df_filo.withColumn("score_local", (0.65 * F.col("part_collectif")) + (0.35 * F.col("part_pauvre")))
df_filo = df_filo.withColumn("score_local", F.least(F.greatest(F.col("score_local"), F.lit(0.0)), F.lit(1.0)))

# Moyenne globale du score local pour la calibration
mean_score = df_filo.select(F.mean("score_local")).first()[0]

# Calibration : On applique le taux de 15% modulé par le score local (qui peut être plus ou moins fort)
df_filo = df_filo.withColumn("taux_sans_voiture_calibre", 
    F.lit(taux_macro_sans_voiture) * (1 + F.col("score_local") - F.lit(mean_score)))
df_filo = df_filo.withColumn("taux_sans_voiture_calibre", F.least(F.greatest(F.col("taux_sans_voiture_calibre"), F.lit(0.0)), F.lit(1.0)))

# Volumes absolus
df_filo = df_filo.withColumn("est_menages_sans_voiture", F.round(F.col("men") * F.col("taux_sans_voiture_calibre"), 2))
df_filo = df_filo.withColumn("est_menages_avec_voiture", F.round(F.col("men") - F.col("est_menages_sans_voiture"), 2))
df_filo = df_filo.withColumn("_ingestion_ts", F.current_timestamp())

# Nettoyage des colonnes temporaires
df_filo = df_filo.drop("total_men", "part_collectif", "part_pauvre", "score_local")

print("✅ Croisement et calibration macro terminés.")

# ==========================================
# C. ENREGISTREMENT
# ==========================================
table_name = "bronze_filosofi_enrichi_voitures"
print("⏳ Écriture de la table Delta finale...")

df_filo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"🎉 Table Delta '{table_name}' enregistrée avec succès !")

# Affichage d'un aperçu
display(spark.table(table_name).select('idcar_200m', 'men', 'ind_2030', 'taux_sans_voiture_calibre', 'est_menages_sans_voiture').limit(5))

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 61, Finished, Available, Finished, False)

⏳ 1. Chargement des tables (Mode PySpark distribué)...
✅ Projections calculées (Taux annuel : 0.2472%)
✅ Taux macro sans voiture (Département 69) fixé à : 15.0%
✅ Croisement et calibration macro terminés.
⏳ Écriture de la table Delta finale...
🎉 Table Delta 'bronze_filosofi_enrichi_voitures' enregistrée avec succès !


SynapseWidget(Synapse.DataFrame, bef5fce6-df9f-44b3-a84d-5efaee0dfa0f)

### 📋 Bilan d'exécution : Ingestion de la Source "Ménages et Voitures" (Couche Bronze)

- **Statut de l'ingestion :** Fichier source chargé avec succès depuis l'espace de stockage OneLake.
- **Persistance Lakehouse :** Table Delta Bronze `menages_voitures_bronze` créée et synchronisée (300 lignes, 4 colonnes)[cite: 4].
- **Analyse et point de vigilance :** Ce dataset contient des statistiques agrégées à l'échelle départementale[cite: 4] (comprenant l'ensemble des départements français). 
- **Prochaine étape méthodologique :** Isoler le département du Rhône (ou la Métropole de Lyon) de cette table pour combiner ces indicateurs de motorisation avec la granularité fine de notre carroyage Filosofi.

### 🚗 Croisement Macro-Micro : Intégration de la Motorisation sur le Carroyage Filosofi

- **Objectif :** Estimer la répartition fine des ménages motorisés et non-motorisés à l'échelle de chaque carreau de 200m pour dimensionner l'offre d'autopartage (Citiz).
- **Défi méthodologique :** Le fichier de motorisation brut fournit une statistique globale pour le département du Rhône (`geocode_departement = '69'`)[cite: 4]. Appliquer cette valeur de manière uniforme à tous les carreaux détruirait la réalité urbaine.
- **Stratégie de pondération intelligente (Micro-localisation) :** 
  1. Extraction de la valeur de référence pour le Rhône depuis la table Bronze `menages_voitures_bronze`[cite: 4].
  2. Utilisation des variables structurelles de la base Filosofi (part de l'habitat collectif `men_coll` et des ménages pauvres `men_pauv`) pour moduler localement le score. Plus un secteur est dense en habitat collectif et urbain, plus la propension de ménages sans voiture est forte.
  3. Génération d'une nouvelle table unifiée et enrichie : `filosofi_enrichi_voiture b


In [44]:
spark.sql("SHOW TABLES").show()

+--------------------+--------------------+-----------+
|           namespace|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|`Cityz Mobilitee_...|bronze_citiz_stat...|      false|
|`Cityz Mobilitee_...|bronze_dott_stations|      false|
|`Cityz Mobilitee_...|bronze_filosofi_e...|      false|
|`Cityz Mobilitee_...|bronze_filosofi_g...|      false|
|`Cityz Mobilitee_...|bronze_filosofi_p...|      false|
|`Cityz Mobilitee_...|bronze_filosofi_r...|      false|
|`Cityz Mobilitee_...|  bronze_irve_bornes|      false|
|`Cityz Mobilitee_...|bronze_leogo_vehi...|      false|
|`Cityz Mobilitee_...|bronze_menages_vo...|      false|
|`Cityz Mobilitee_...|bronze_parkings_o...|      false|
|`Cityz Mobilitee_...|bronze_poi_commerces|      false|
|`Cityz Mobilitee_...|    bronze_poi_sante|      false|
|`Cityz Mobilitee_...|   bronze_poi_sirene|      false|
|`Cityz Mobilitee_...|   bronze_pvo_voirie|      false|
|`Cityz Mobilitee_...|   bronze_tcl_arrets|     

### 📋 Bilan d'exécution : Génération de la Table Maître "Filosofi Enrichi & Motorisation"

- **Statut des calculs :** Traitement Spark exécuté avec succès (10 de 10 tâches réussies)[cite: 5]. 
- **Persistance Lakehouse :** Table Delta finale `filosofi_enrichi_voitures` enregistrée et synchronisée avec brio[cite: 5].
- **Résultats intégrés dans la table :** 
  * Conservation de la maille fine de 200m (`idcar_200m`) et du nombre total de ménages (`men`)[cite: 5].
  * Modélisation fine de la répartition des ménages sans voiture (`est_menages_sans_voiture`) et avec voiture (`est_menages_avec_voiture`) via le score de propension structurelle[cite: 5].
- **Prochaine étape :** Ce référentiel unique centralise désormais la démographie actuelle, les projections à long terme (2030, 2040, 2050) et les indicateurs de mobilité, constituant la base parfaite pour la cartographie géomarketing et l'implantation des stations Citiz.

### 🏢 Ingestion & Cadrage Méthodologique : Points d'Intérêt (POI) Stratégiques & Économiques

- **Objectif métier :** Cartographier les grands pôles générateurs de flux quotidiens (emploi, soins, commerce) afin d'identifier les zones d'attractivité à fort potentiel pour l'implantation des stations d'autopartage Citiz.
- **Types de POI ciblés :**
  1. **Tissu Économique & Emploi (Base SIRENE / INSEE) :** Densité des établissements actifs et grands centres d'affaires.
  2. **Établissements de Santé & Hospitaliers :** Hôpitaux, cliniques et grands centres de soins (générateurs de déplacements pendulaires et de visiteurs).
  3. **Zones Commerçantes & Enseignes Majeures :** Pôles commerciaux de proximité et centres d'attractivité locale.
- **Stratégie d'ingestion (Couche Bronze) :** Interrogation directe et automatisée des API publiques (API SIRENE / Overpass OSM / OpenData) avec filtrage géofensé sur le territoire de la Métropole de Lyon, puis persistance au format Delta dans le Lakehouse (`poi_economiques_bronze`, `poi_sante_bronze`).
- **Point de vigilance :** Nettoyage des données pour exclure les établissements inactifs ou administratifs sans accueil physique de public.


### 🏢 Analyse et Spécifications de la Source : Base SIRENE (INSEE)

- **Fournisseur / Opérateur :** INSEE (Institut National de la Statistique et des Études Économiques), via data.gouv.fr.
- **URL de la ressource :** `https://www.data.gouv.fr/fr/datasets/r/a28527c0-0221-41f5-a230-bbb1d50a5392` (Fichier StockEtablissement - base nationale).
- **Format d'entrée :** Fichier CSV (séparateur `,`, très volumineux, plusieurs gigaoctets).
- **Indicateurs de fraîcheur :** Mise à jour mensuelle par l'INSEE (le fichier stock est écrasé à chaque nouvelle publication).

#### 🌳 Topographie et Structure du Payload
- **Format :** Fichier tabulaire classique (pas de JSON imbriqué).
- **Attributs clés (parmi des dizaines) :**
  * `siren`, `siret` : identifiants uniques de l'entreprise et de l'établissement.
  * `codeCommuneEtablissement` / `codePostalEtablissement` : localisation géographique (ex. `69123` pour Lyon).
  * `libelleCommuneEtablissement` : nom de la commune.
  * `activitePrincipaleEtablissement` : code NAF/APE (ex. `62.01Z` pour la programmation).
  * `etablissementSiege` : indique si c'est le siège social.

#### ⚠️ Points de Vigilance Techniques (Couche Bronze) :
- **Volumétrie critique :** Le fichier national contient plus de **10 millions de lignes**. Un `pd.read_csv()` classique sature la mémoire du driver Spark. 
- **Stratégie de filtrage (Streaming) :** Nous allons utiliser le paramètre `chunksize` de Pandas pour lire le fichier par blocs de 100 000 lignes, et **ne conserver que les établissements du Rhône** (code postal ou code commune commençant par `69`). Cela réduit la table finale à ~200 000 lignes, sans saturer la mémoire.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** Requête HTTP GET en streaming avec filtrage à la volée (`chunksize`).
- **Stockage cible :** Table Delta `bronze_poi_sirene` (nommage conforme).


In [50]:
import requests
import pandas as pd
import io

url_direct = "https://www.data.gouv.fr/fr/datasets/r/a28527c0-0221-41f5-a230-bbb1d50a5392"

print("⏳ Téléchargement direct du flux SIRENE en mémoire...")

try:
    # 1. Requête HTTP
    response = requests.get(url_direct, timeout=(10, 60))
    response.raise_for_status() 
    print(f"🎉 Téléchargement réussi ! ({len(response.content) / 1024 / 1024:.2f} Mo).")

    # 2. Lecture CSV
    csv_content = response.content.decode('utf-8', errors='ignore')
    pdf_temp = pd.read_csv(io.StringIO(csv_content), sep=None, engine='python')
    print(f"📊 {len(pdf_temp)} lignes et {len(pdf_temp.columns)} colonnes détectées.")

    # 3. Traçabilité (Couche Bronze)
    pdf_temp["_source_url"] = url_direct
    pdf_temp["_ingestion_ts"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

    # 4. Écriture Spark DIRECTE (comme dans ton notebook d'origine)
    print("⏳ Écriture de la table Delta 'bronze_poi_sirene'...")
    spark_df_sirene = spark.createDataFrame(pdf_temp.astype(str))
    
    table_name_sirene = "bronze_poi_sirene"
    spark_df_sirene.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name_sirene)

    print(f"✅ Table Delta '{table_name_sirene}' enregistrée avec succès !")
    display(pdf_temp.head(5))

except Exception as e:
    print(f"❌ Erreur lors de l'ingestion SIRENE : {e}")

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 63, Finished, Available, Finished, False)

⏳ Téléchargement direct du flux SIRENE en mémoire...
🎉 Téléchargement réussi ! (94.38 Mo).
📊 324971 lignes et 18 colonnes détectées.
⏳ Écriture de la table Delta 'bronze_poi_sirene'...
✅ Table Delta 'bronze_poi_sirene' enregistrée avec succès !


SynapseWidget(Synapse.DataFrame, 1b52a505-8cbc-407c-9afa-d761563818e0)

### 🏥 Analyse de la Source & Cadrage Métier : Établissements de Santé (Hôpitaux)

- **Objectif Métier & Utilité de la Source :** Les pôles hospitaliers et grands centres de santé de la Métropole de Lyon constituent des concentrateurs massifs de flux quotidiens (personnel soignant aux horaires atypiques, patients, visiteurs). L'intégration de ces données dans notre Lakehouse est stratégique pour le projet Citiz : elle permet d'identifier des zones de forte attraction où le besoin d'autopartage en station fixe répond directement à des problématiques de mobilité pendulaire et d'accessibilité aux soins.
- **Origine & Caractéristiques de la Source :** Flux WFS officiel de la Métropole de Lyon (`adr_voie_lieu.adrhopital`), diffusant les emprises géographiques et les attributs administratifs des structures sanitaires du territoire.
- **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium.
- **Points de Vigilance :**
  * **Hétérogénéité géométrique :** Présence de structures topologiques complexes (polygones, multipolygones ou points) qu'il conviendra de traiter spécifiquement lors du passage à la couche Silver. Pour la couche Bronze, la géométrie brute est sérialisée en chaîne de caractères (`geometry_raw`) pour garantir l'intégrité.
  * **Intégrité de la donnée brute :** Respect strict du principe d'ingestion "tel quel" (zéro perte d'attributs d'origine, conservation de la structure JSON initiale du flux) pour garantir la traçabilité et l'audibilité du pipeline.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requête HTTP GET sur le flux WFS, avec `requests.get(timeout=TIMEOUT_REQUETE)`, `raise_for_status()` bloquant et retry borné.
- **Stockage cible :** Table Delta `bronze_poi_sante` (nommage conforme).


In [51]:
import requests
import pandas as pd
import time

# URL officielle du flux WFS des hôpitaux de la Métropole de Lyon (EPSG:4326)
url_hopitaux = (
    "https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=metropole-de-lyon:adr_voie_lieu.adrhopital"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

print("⏳ Interrogation brute du flux WFS des hôpitaux de Lyon...")

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_hopitaux, timeout=(10, 60))
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_hopitaux}")

data_json = response.json()
features = data_json.get("features", [])

# Ingestion brute : conservation intégrale des propriétés et de la géométrie source
records = []
for feat in features:
    props = feat.get("properties", {}).copy()
    # On stocke la géométrie brute sous forme de chaîne pour garantir l'intégrité
    props['geometry_raw'] = str(feat.get("geometry", {}))
    records.append(props)

df_hopitaux_brut = pd.DataFrame(records)

# Ajout des colonnes de traçabilité
df_hopitaux_brut["_source_url"]          = url_hopitaux
df_hopitaux_brut["_source_last_updated"] = data_json.get("timeStamp")
df_hopitaux_brut["_source_ttl"]          = None
df_hopitaux_brut["_source_version"]      = data_json.get("numberMatched")
df_hopitaux_brut["_ingestion_ts"]        = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"🎉 Succès ! {len(df_hopitaux_brut)} enregistrements bruts chargés. Enregistrement en cours...")

# Enregistrement direct et sans altération dans la table Delta Bronze
spark_df_hopitaux = spark.createDataFrame(df_hopitaux_brut.astype(str))
table_name_hopitaux = "bronze_poi_sante"

spark_df_hopitaux.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name_hopitaux)

print(f"📦 Table Delta Bronze '{table_name_hopitaux}' enregistrée et synchronisée avec succès dans le Lakehouse !")

# Aperçu de contrôle de la structure brute
display(df_hopitaux_brut.head(5))

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 66, Finished, Available, Finished, False)

⏳ Interrogation brute du flux WFS des hôpitaux de Lyon...
🎉 Succès ! 54 enregistrements bruts chargés. Enregistrement en cours...
📦 Table Delta Bronze 'bronze_poi_sante' enregistrée et synchronisée avec succès dans le Lakehouse !


SynapseWidget(Synapse.DataFrame, 9cab1a5b-82a9-42ac-a635-65802e604bae)

### 🛍️ Analyse de la Source & Cadrage Métier : Centres Commerciaux & Pôles d'Attractivité (Métropole de Lyon)

- **Objectif Métier & Utilité de la Source :** Les centres commerciaux et grands pôles d'achats de la Métropole de Lyon constituent des aimants majeurs de flux de chalandise et de déplacements non pendulaires (consommation, loisirs, emploi tertiaire). Leur intégration dans notre pipeline Lakehouse est indispensable pour dimensionner l'offre de mobilité partagée Citiz sur des zones à forte rotation de véhicules et à enjeux commerciaux prononcés.
- **Origine & Caractéristiques de la Source :** Flux WFS officiel de la Métropole de Lyon (`adr_voie_lieu.adrcentrecommerc_latest`), recensant les emprises et les attributs descriptifs des zones commerciales et galeries marchandes du territoire.
- **Système de coordonnées demandé :** `EPSG:4326` (WGS84, degrés) — aligné avec le stockage Bronze et Folium.
- **Points de Vigilance :**
  * **Volatilité et périmètre :** Suivi des versions actualisées de la base (`_latest`) pour refléter les ouvertures ou restructurations récentes des surfaces commerciales.
  * **Intégrité de la donnée brute :** Application stricte du principe d'ingestion de la couche Bronze (conservation intégrale des attributs d'origine et de la géométrie native sous forme de chaîne `geometry_raw` sans altération ni transformation en amont).

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** requête HTTP GET sur le flux WFS, avec `requests.get(timeout=(10, 60))`, `raise_for_status()` bloquant et retry borné.
- **Stockage cible :** Table Delta `bronze_poi_commerces` (nommage conforme).


In [52]:
import requests
import pandas as pd
import time

# URL officielle du flux WFS des centres commerciaux (EPSG:4326)
url_commerces = (
    "https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=metropole-de-lyon:adr_voie_lieu.adrcentrecommerc_latest"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

print("⏳ Interrogation brute du flux WFS des centres commerciaux de Lyon...")

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_commerces, timeout=(10, 60))
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_commerces}")

data_json = response.json()
features = data_json.get("features", [])

# Ingestion brute : conservation intégrale des propriétés et de la géométrie source
records = []
for feat in features:
    props = feat.get("properties", {}).copy()
    # Stockage de la géométrie brute pour respecter le principe d'intégrité de la couche Bronze
    props['geometry_raw'] = str(feat.get("geometry", {}))
    records.append(props)

df_commerces_brut = pd.DataFrame(records)

# Ajout des colonnes de traçabilité (Règle d'or de la Couche Bronze)
df_commerces_brut["_source_url"]          = url_commerces
df_commerces_brut["_source_last_updated"] = data_json.get("timeStamp")
df_commerces_brut["_source_ttl"]          = None
df_commerces_brut["_source_version"]      = data_json.get("numberMatched")
df_commerces_brut["_ingestion_ts"]        = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"🎉 Succès ! {len(df_commerces_brut)} enregistrements bruts chargés. Enregistrement en cours...")

# Enregistrement direct et sans altération dans la table Delta Bronze
spark_df_commerces = spark.createDataFrame(df_commerces_brut.astype(str))
table_name_commerces = "bronze_poi_commerces"

spark_df_commerces.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name_commerces)

print(f"📦 Table Delta Bronze '{table_name_commerces}' enregistrée et synchronisée avec succès dans le Lakehouse !")

# Aperçu de contrôle de la structure brute
display(df_commerces_brut.head(5))

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 69, Finished, Available, Finished, False)

⏳ Interrogation brute du flux WFS des centres commerciaux de Lyon...
🎉 Succès ! 29 enregistrements bruts chargés. Enregistrement en cours...
📦 Table Delta Bronze 'bronze_poi_commerces' enregistrée et synchronisée avec succès dans le Lakehouse !


SynapseWidget(Synapse.DataFrame, 8ec71ac7-a91b-4f7d-8e06-d93fcad10a5f)

## 🌍 5. Réglementation, Écologie & Zones à Faibles Émissions (ZFE)

### 🎯 Objectif Stratégique & Rôle dans le Modèle Citiz
L'intégration de la brique réglementaire et écologique (et en premier lieu le périmètre des **Zones à Faibles Émissions - ZFE** de la Métropole de Lyon) constitue l'élément d'arbitrage macro-environnemental de notre Lakehouse. 

Pour un service d'autopartage comme Citiz, la ZFE ne se résume pas à une simple contrainte juridique : c'est un **accélérateur structurel de report modal**. En restreignant progressivement l'accès du territoire aux véhicules thermiques les plus anciens (calendrier des vignettes Crit'Air), la politique ZFE modifie en profondeur les arbitrages de mobilité des ménages et des professionnels lyonnais.

### 🧩 Croisement Analytique & Valeur Ajoutée pour le Lakehouse
- **Couplage avec la démographie (Filosofi) :** Identifier les zones résidentielles où la pression réglementaire est forte mais où l'équipement automobile des ménages reste fragile, révélant un besoin critique d'alternatives partagées.
- **Optimisation du maillage de flotte :** Positionner ou dimensionner les stations et les véhicules (notamment propres/électriques) à la charnière des frontières de la ZFE pour capter les reports de trafic et faciliter les interconnexions avec les transports en commun (TCL).
- **Complétude méthodologique :** Boucler notre catalogue de données en croisant l'offre (TCL, micromobilité, autopartage), la demande (foyers, POI économiques/santé) et le cadre légal qui contraint l'espace routier.


### 🔍 Analyse détaillée, Intérêt Stratégique et Points de Vigilance : Périmètre ZFE (Zones à Faibles Émissions - Métropole de Lyon)

- **Fournisseur / Opérateur :** Métropole de Lyon (Direction de l'Écologie et des Déplacements)
- **URL de l'API / Flux :** `https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?SERVICE=WFS&VERSION=2.0.0&request=GetFeature&typename=metropole-de-lyon:eco_ecologie.zfe&outputFormat=application/json&SRSNAME=EPSG:4171`
- **Format d'entrée :** GeoJSON / OGC API Features (Flux géographique de polygones représentant le périmètre réglementaire de la Zone à Faibles Émissions)
- **Indicateurs de fraîcheur et Gouvernance :**
  * Données fondamentales reflétant le cadre juridique et les restrictions progressives de circulation des véhicules polluants (vignettes Crit'Air) sur le territoire métropolitain.
  * Évolution réglementaire nécessitant un suivi de version pour coller aux calendriers de durcissement des interdictions de circulation.

#### 🌳 Topographie et Structure du Payload JSON
- **Chemin racine des données :** Objet global GeoJSON contenant un tableau `features`.
- **Structure d'un objet `Feature` :**
  * `id` ou `type` : Identifiant unique de l'entité spatiale (ex: `eco_ecologie.zfe.200046977-ZFE-001`).
  * `geometry` : Objet géométrique de type `Polygon` (ou `MultiPolygon`) avec ses coordonnées de contours denses en système `EPSG:4171`.
  * `properties` : Contient l'ensemble des attributs descriptifs liés à la réglementation, aux dérogations ou aux libellés du zonage.

#### 📦 Concrètement, que regroupe ce jeu de données ?
D'après l'analyse du flux et de sa structure attributaire, ce dataset regroupe **l'emprise géographique officielle de la ZFE de la Métropole de Lyon**, délimitant l'espace où s'appliquent les restrictions de circulation environnementales. Il centralise :
- **L'enveloppe spatiale du zonage :** Les frontières précises qui séparent les zones soumises à restriction de celles qui ne le sont pas.
- **Les attributs réglementaires associés :** Les métadonnées décrivant le périmètre et son cadre juridique d'application.

#### 🚀 Comment allons-nous l'utiliser dans notre projet de géomarketing ?
Nous allons exploiter ce flux pour intégrer la contrainte macro-environnementale dans notre modèle d'autopartage Citiz :
1. **Croisement socio-démographique (Filosofi) :** Identifier les populations résidentes situées à l'intérieur ou à la frontière immédiate de la ZFE pour mesurer l'exposition des ménages aux interdictions de véhicules thermiques et quantifier le potentiel de report vers l'autopartage.
2. **Optimisation du maillage de flotte et des stations :** Positionner stratégiquement nos stations aux portes de la ZFE ou à l'intérieur pour capter les usagers cherchant une alternative propre pour leurs déplacements urbains.
3. **Analyse d'accessibilité réglementée :** Combiner le périmètre avec nos analyses d'infrastructures et de flux pour évaluer l'attractivité de l'offre partagée face aux contraintes de circulation.

#### 🛡️ Règle d'Ingestion (Couche Bronze) :
- **Mode de collecte :** Requête HTTP GET adaptée au standard WFS/GeoJSON (`outputFormat=application/json`, `SRSNAME=EPSG:4171`).
- **Stockage cible :** Table Delta `zfe_perimetre_bronze` pour préserver l'état brut des géométries de type `Polygon` et des attributs sans transformation prématurée.


In [53]:
import requests
import pandas as pd
import time

# URL officielle du flux WFS de la ZFE de la Métropole de Lyon (EPSG:4326, count=5000)
url_zfe = (
    "https://data.grandlyon.com/geoserver/metropole-de-lyon/ows?"
    "SERVICE=WFS&VERSION=2.0.0&request=GetFeature"
    "&typename=metropole-de-lyon:eco_ecologie.zfe"
    "&outputFormat=application/json&SRSNAME=EPSG:4326"
    "&startIndex=0&count=5000&sortby=gid"
)

print("⏳ Interrogation brute du flux WFS du périmètre ZFE de Lyon...")

MAX_RETRIES = 3
for tentative in range(MAX_RETRIES):
    try:
        response = requests.get(url_zfe, timeout=(10, 60))
        response.raise_for_status()
        break
    except requests.exceptions.Timeout:
        print(f"⚠️ Timeout (tentative {tentative+1}/{MAX_RETRIES})")
        time.sleep(2)
else:
    raise RuntimeError(f"Échec après {MAX_RETRIES} tentatives sur {url_zfe}")

data_json = response.json()
features = data_json.get("features", [])

# Ingestion brute : conservation intégrale des propriétés et de la géométrie source
records = []
for feat in features:
    props = feat.get("properties", {}).copy()
    # Stockage de la géométrie brute (polygones complexes) pour respecter l'intégrité Bronze
    props['geometry_raw'] = str(feat.get("geometry", {}))
    props['feature_id'] = feat.get("id")
    records.append(props)

df_zfe_brut = pd.DataFrame(records)

# Ajout des colonnes de traçabilité (Règle d'or de la Couche Bronze)
df_zfe_brut["_source_url"]          = url_zfe
df_zfe_brut["_source_last_updated"] = data_json.get("timeStamp")
df_zfe_brut["_source_ttl"]          = None
df_zfe_brut["_source_version"]      = data_json.get("numberMatched")
df_zfe_brut["_ingestion_ts"]        = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"🎉 Succès ! {len(df_zfe_brut)} entités de périmètre ZFE chargées. Enregistrement en cours...")

# Enregistrement direct dans la table Delta Bronze (nommage conforme)
spark_df_zfe = spark.createDataFrame(df_zfe_brut.astype(str))
table_name_zfe = "bronze_zfe_perimetre"

spark_df_zfe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name_zfe)

print(f"📦 Table Delta Bronze '{table_name_zfe}' enregistrée et synchronisée avec succès dans le Lakehouse !")

# Aperçu de contrôle de la structure brute
display(df_zfe_brut.head(5))

StatementMeta(, d3a0e1b2-9b43-4c0c-8063-83ab69f7e7b5, 72, Finished, Available, Finished, False)

⏳ Interrogation brute du flux WFS du périmètre ZFE de Lyon...
🎉 Succès ! 1 entités de périmètre ZFE chargées. Enregistrement en cours...
📦 Table Delta Bronze 'bronze_zfe_perimetre' enregistrée et synchronisée avec succès dans le Lakehouse !


SynapseWidget(Synapse.DataFrame, 3bff92c8-d304-4091-a267-9ad9564931e7)